# NucleiSky2D Benchmarking

## 🎯 Objective
The goal of this notebook is to determine the **minimal number of nuclei** (or minimal patch size) required to reliably locate a cropped image within a large reference image ("The Haystack").

This quantifies the robustness of matching algorithms (Graph, Quad, Triangles, Hashing) and answers the question: *How much biological context is needed to fingerprint a specific tissue region?*

## ⚙️ Workflow
1.  **Select Reference Image**: Load a full slide or large field-of-view image.
2.  **Segmentation**: Detect nuclei and extract centroids/features from the full image.
3.  **Synthetic Benchmarking**:
    * The system generates random "virtual crops" of varying sizes from the full image.
    * It filters for crops containing specific numbers of nuclei.
    * It attempts to match these crops back to the full image using various algorithms.
4.  **Analysis**:
    * Calculate the success rate as a function of nuclei count ($N$) and patch size.
    * Fit a logistic regression (sigmoid) curve to the binary success data to determine the precise, continuous threshold where re-identification probability reaches the 90% target.
    * Evaluate computational efficiency by tracking matching speed against the number of nuclei.

## 🔑 Key Metrics
* **Unified Success**: A match is considered successful *only* if it simultaneously meets four strict criteria:
    1.  The base algorithm reports a successful match.
    2.  **Translation Error**: The estimated position is within a small margin of error (e.g., < 20 px) of the true ground-truth coordinate.
    3.  **Inlier Fraction**: At least 80% of the nuclei in the crop align mathematically with the reference map.
    4.  **SSIM (Structural Similarity)**: The matched image region achieves an SSIM of at least 0.80 compared to the ground truth.
* **$N_{min}$**: The smallest number of nuclei required to achieve a $\ge$ 90% success rate, derived from the sigmoid curve fit.
* **Execution Speed**: The total time in seconds required to compute the match, evaluating how each algorithm scales with increased biological context.

# 0. Install + import functions

In [ ]:
# @title  Initialize Environment & Install Dependencies

from IPython.display import display
import torch
import sys

# Metadata
current_version = "0.0.6"
notebook_name = "NucleiSky2DBenchmarking"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------

if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    !pip install -q "cellpose[all]" tifffile
    !pip install -q instanseg-torch

    from google.colab import userdata

    # 1. Retrieve the token securely from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        print("Error: Secret 'GITHUB_TOKEN' not found. Please add it in the sidebar (🔑).")
        token = None

    # 2. Install from your private repo
    if token:
        user = 'CellMigrationLab'  # Replace with your GitHub username
        repo = 'NucleiSky'

        # The --upgrade flag ensures you always get the latest version if you re-run
        !pip install --upgrade git+https://{token}@github.com/{user}/{repo}.git

    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")

else:
    # Fallback for local environments
    print("done")


# ---------------------------------------------------------
# 2. GPU Check
# ---------------------------------------------------------
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  WARNING: No GPU detected. Segmentation with Cellpose or InstanSeg will be VERY SLOW on CPU.")
    print("    -> In Colab, go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")



In [ ]:
#@title Imports and Benchmark functions

# ---------------------------------------------------------
# 3. Main Imports
# ---------------------------------------------------------

from IPython.display import display, clear_output
from tifffile import imwrite, imread
import ipywidgets as widgets
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import html
import copy
import traceback

import os
import time
import zlib

import pandas as pd

from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as ssim

from nucleisky.nucleisky2d.demo_utils import generate_random_crop

# IO
from nucleisky.nucleisky2d.io import (
    load_image,
    load_transforms_any,
    get_pixel_size_um_from_tiff,
    save_json,
    make_result_dir,
    save_nucleisky_transform,
    append_transform_jsonl
)

# Preprocess
from nucleisky.nucleisky2d.preprocess import (
    require_2d_image,
    require_2d_label_mask,
    require_positive_float,
    scale_normalize_pair_for_segmentation,
    ij_percentile_normalize
)


# 1. Pure Export
from nucleisky.nucleisky2d.export import (
    export_aligned_imagej_stacks,
    run_export_with_record,
    inspect_image_header,
)

# 2. Visualization
from nucleisky.nucleisky2d.visualization import (
    imshow_safe,
    show_alignment_original_and_rescaled,
    downsample_points_for_display
)

# -----------------------------------------------

from nucleisky.nucleisky2d.segmentation import segment_nuclei_dispatch
from nucleisky.nucleisky2d.features import (
    extract_nuclear_features,
    add_centroids_orig_px_columns,
    centroids_from_df,
    extract_centroids_um,
    stack_feature_vectors
)
from nucleisky.nucleisky2d.config import save_matcher_config_json, DEFAULT_MATCHER_CONFIG
from nucleisky.nucleisky2d.pipeline import (
    NucleiSky,
    run_adaptive_matching_and_export,
    pick_best_transform,
    run_adaptive_nucleisky
)

print("✅ Environment Ready")

BENCH_COLS = [
    "matcher",
    "patch_h_px", "patch_w_px",
    "patch_y0", "patch_x0",
    "n_nuclei",
    "success",
    "frac_inliers", "mean_error_um", "trans_error_px", "ssim",
    "time_matcher_s", "time_total_s",
]

def _sanitize_controller_kwargs(d: dict | None) -> dict:
    """
    Allow ONLY keys that run_adaptive_nucleisky actually expects as controller params.
    Everything else is dropped to avoid accidentally forwarding matcher kwargs (e.g. "_common").
    """
    if d is None:
        return {}

    if not isinstance(d, dict):
        raise ValueError("For matcher='adaptive', matcher_kwargs must be a dict of controller options.")

    allowed = {
        "matcher_order",
        "base_seed",
        "matcher_config",
        "stop_on_success",
        "store_full_out",
        "max_total_time_s",
        "min_frac_inliers_success",
        "max_mean_error_um_success",
        "accept_transform_without_success",
    }

    out = {k: v for k, v in d.items() if k in allowed}
    dropped = [k for k in d.keys() if k not in allowed]
    if dropped:
        print(f"ℹ️ adaptive: dropped unsupported controller kwargs: {dropped}")
    return out



def _clip_origin(y0: int, x0: int, H: int, W: int, ph: int, pw: int) -> tuple[int, int]:
    y0 = int(max(0, min(int(y0), int(H - ph))))
    x0 = int(max(0, min(int(x0), int(W - pw))))
    return y0, x0


def _count_nuclei_in_patch(ys_px: np.ndarray, xs_px: np.ndarray, y0: int, x0: int, ph: int, pw: int) -> int:
    y1 = y0 + ph
    x1 = x0 + pw
    return int(np.sum((ys_px >= y0) & (ys_px < y1) & (xs_px >= x0) & (xs_px < x1)))


def make_patch_candidates_anchored_on_nuclei(
    H: int,
    W: int,
    patch_h_px: int,
    patch_w_px: int,
    ys_px: np.ndarray,
    xs_px: np.ndarray,
    max_trials: int,
    seed: int,
    min_nuclei: int = 1,
    oversample_factor: int = 25,
) -> np.ndarray:
    """
    Deterministically propose up to max_trials patch origins (y0,x0) such that
    each patch contains at least min_nuclei nuclei (based on ys_px/xs_px).
    """
    patch_h_px = int(patch_h_px); patch_w_px = int(patch_w_px)
    max_trials = int(max_trials)
    min_nuclei = int(min_nuclei)

    if H <= patch_h_px or W <= patch_w_px:
        return np.zeros((0, 2), dtype=int)

    ys_px = np.asarray(ys_px, dtype=int)
    xs_px = np.asarray(xs_px, dtype=int)
    if ys_px.size == 0:
        return np.zeros((0, 2), dtype=int)

    rng = np.random.default_rng(int(seed))

    # We try more proposals than needed, then keep the first max_trials that satisfy min_nuclei.
    n_proposals = max(max_trials * int(oversample_factor), max_trials)
    idx = rng.integers(0, len(ys_px), size=n_proposals, endpoint=False)

    out = []
    seen = set()

    for i in idx:
        y = int(ys_px[i])
        x = int(xs_px[i])

        # Place the chosen nucleus at a random location inside the patch
        oy = int(rng.integers(0, patch_h_px))
        ox = int(rng.integers(0, patch_w_px))

        y0, x0 = _clip_origin(y - oy, x - ox, H, W, patch_h_px, patch_w_px)

        if (y0, x0) in seen:
            continue

        if min_nuclei > 1:
            n_in = _count_nuclei_in_patch(ys_px, xs_px, y0, x0, patch_h_px, patch_w_px)
            if n_in < min_nuclei:
                continue

        seen.add((y0, x0))
        out.append((y0, x0))
        if len(out) >= max_trials:
            break

    return np.asarray(out, dtype=int)



def _stable_u32_from_str(s: str) -> int:
    """Stable across Python sessions (unlike built-in hash())."""
    return zlib.adler32(s.encode("utf-8")) & 0xFFFFFFFF


import copy

def _deep_merge_dict(base: dict, override: dict | None) -> dict:
    """Recursively merge override into base without mutating either."""
    if override is None:
        return copy.deepcopy(base)
    out = copy.deepcopy(base)
    for k, v in override.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge_dict(out[k], v)
        else:
            out[k] = copy.deepcopy(v)
    return out


def make_patch_candidates(H, W, patch_h_px, patch_w_px, max_trials, seed, stride_frac=0.5):
    """
    Return up to max_trials candidate (y0,x0) patch origins.
    Deterministic for a given (H,W,patch_h_px,patch_w_px,seed,stride_frac).
    """
    patch_h_px = int(patch_h_px); patch_w_px = int(patch_w_px)
    if H <= patch_h_px or W <= patch_w_px:
        return np.zeros((0, 2), dtype=int)

    stride_y = max(1, int(round(patch_h_px * float(stride_frac))))
    stride_x = max(1, int(round(patch_w_px * float(stride_frac))))

    y_candidates = np.arange(0, H - patch_h_px + 1, stride_y, dtype=int)
    x_candidates = np.arange(0, W - patch_w_px + 1, stride_x, dtype=int)

    Y0, X0 = np.meshgrid(y_candidates, x_candidates, indexing="ij")
    candidates = np.column_stack([Y0.ravel(), X0.ravel()]).astype(int, copy=False)

    rng = np.random.default_rng(int(seed))
    rng.shuffle(candidates)

    if len(candidates) > int(max_trials):
        candidates = candidates[: int(max_trials)]
    return candidates


def make_benchmark_patch_plan(
    img_full,
    patch_sizes_px,
    patches_per_size,
    seed=0,
    stride_frac=0.5,   # kept for compatibility; not used in anchored mode
    df_full=None,
    min_nuclei=1,      # set to 3 to avoid your n_nuc<3 early skip
):
    """
    Returns dict: {size_px: candidates_array(N,2) of (y0,x0)}.

    If df_full is provided, candidates are anchored on nuclei and filtered to contain >= min_nuclei nuclei.
    Deterministic per (seed, size).
    """
    img_full = np.asarray(img_full)
    if img_full.ndim != 2:
        raise ValueError(f"Expected 2D image. Got shape={img_full.shape}")

    H, W = img_full.shape
    plan = {}

    # Precompute nuclei coords (px) if provided
    ys_px = xs_px = None
    if df_full is not None:
        if ("centroid_y_px" not in df_full.columns) or ("centroid_x_px" not in df_full.columns):
            raise ValueError("df_full must have centroid_y_px and centroid_x_px to enforce min_nuclei.")
        ys_px = df_full["centroid_y_px"].to_numpy(dtype=int, copy=False)
        xs_px = df_full["centroid_x_px"].to_numpy(dtype=int, copy=False)

    for size in patch_sizes_px:
        size = int(size)
        ss = np.random.SeedSequence([int(seed), int(size)])
        size_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        if df_full is not None:
            cand = make_patch_candidates_anchored_on_nuclei(
                H, W,
                patch_h_px=size,
                patch_w_px=size,
                ys_px=ys_px,
                xs_px=xs_px,
                max_trials=int(patches_per_size),
                seed=int(size_seed),
                min_nuclei=int(min_nuclei),
            )
        else:
            # fallback to your original geometric plan
            cand = make_patch_candidates(
                H, W,
                patch_h_px=size,
                patch_w_px=size,
                max_trials=int(patches_per_size),
                seed=int(size_seed),
                stride_frac=float(stride_frac),
            )

        plan[size] = cand

    return plan



def _benchmark_runtime_overrides(matcher: str, base_overrides: dict | None, run_seed: int) -> dict:
    """
    Ensure NucleiSky receives deterministic random_state=run_seed,
    regardless of user overrides.
    Accepts:
      - hierarchical: {"_common": {...}, "graph": {...}}
      - flat: {"n_iters": 50000, ...} (assumed matcher-specific)
    """
    matcher_mode = str(matcher).strip().lower()
    out = {"_common": {}, matcher_mode: {}}

    if base_overrides:
        if not isinstance(base_overrides, dict):
            raise ValueError("matcher_kwargs must be a dict (flat or hierarchical).")

        is_hier = ("_common" in base_overrides) or (matcher_mode in base_overrides)
        if is_hier:
            out = _deep_merge_dict(out, base_overrides)
        else:
            out[matcher_mode] = dict(base_overrides)

    # ALWAYS enforce the benchmark seed last
    out.setdefault("_common", {})
    out["_common"]["random_state"] = int(run_seed)
    return out


def _extract_quality_safe(out: dict) -> tuple[float, float | None]:
    q = out.get("match_quality", {}) or {}
    try:
        frac = float(q.get("frac_inliers", 0.0) or 0.0)
    except Exception:
        frac = 0.0
    err = q.get("mean_error_um", None)
    return frac, err


def run_single_patch_match_benchmark(
    img_full,
    df_full,
    PIXEL_SIZE_UM,
    patch_h_px,
    patch_w_px,
    max_trials=50,
    random_state=None,
    compute_ssim=True,
    matcher="graph",

    matcher_config=None,
    matcher_kwargs=None,

    save_path=None,
    restart=False,
    candidates=None,
    candidates_seed=None,
    stride_frac=0.5,
    ij_percentile_normalize=None,
):
    """
    Benchmarks a single matcher on a set of candidate patches.
    UPDATED: ij_percentile_normalize restored (required), do_plots removed (unexpected).
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize (used for SSIM calculation).")

    img_full = np.asarray(img_full)
    if img_full.ndim != 2:
        raise ValueError(f"Benchmark expects a 2D image. Got shape={img_full.shape}")

    H, W = img_full.shape
    patch_h_px = int(patch_h_px); patch_w_px = int(patch_w_px)

    if H <= patch_h_px or W <= patch_w_px:
        print(f"Patch {patch_h_px}x{patch_w_px} is larger than image {H}x{W}, skipping.")
        return []

    matcher_mode = str(matcher).strip().lower()

    # ---- Candidates (fixed across matchers if provided) ----
    if candidates is None:
        seed = int(candidates_seed if candidates_seed is not None else (random_state if random_state is not None else 0))
        candidates = make_patch_candidates(
            H, W,
            patch_h_px=patch_h_px,
            patch_w_px=patch_w_px,
            max_trials=int(max_trials),
            seed=seed,
            stride_frac=float(stride_frac),
        )
    candidates = np.asarray(candidates, dtype=int)
    if candidates.ndim != 2 or candidates.shape[1] != 2:
        raise ValueError(f"candidates must be (N,2) array of (y0,x0). Got {candidates.shape}")

    # ---- df requirements ----
    required_cols = ["centroid_y_um", "centroid_x_um", "centroid_y_px", "centroid_x_px"]
    missing_cols = [c for c in required_cols if c not in df_full.columns]
    if missing_cols:
        raise ValueError(f"df_full missing required columns: {missing_cols}")

    # For "graph" we require feature_vector
    if matcher_mode == "graph":
        if "feature_vector" not in df_full.columns:
            raise ValueError("df_full missing required column for graph matcher: 'feature_vector'")

    # Precompute full centroids once
    centroids_full_um = df_full[["centroid_y_um", "centroid_x_um"]].to_numpy(dtype=float, copy=False)

    # Precompute full features if available and needed
    has_feat = ("feature_vector" in df_full.columns) and (len(df_full) > 0)
    features_full = None
    if has_feat and (matcher_mode in ("graph", "adaptive")):
        features_full = np.stack(df_full["feature_vector"].to_numpy())

    # ---- checkpointing ----
    results = []
    already_done_coords = set()

    if save_path:
        save_path = str(save_path)
        if restart and os.path.exists(save_path):
            os.remove(save_path)
            print(f"Restart=True → deleted checkpoint: {save_path}")

        if (not restart) and os.path.exists(save_path):
            print(f"Resuming from checkpoint: {save_path}")
            try:
                prev_df = pd.read_csv(save_path)
                if ("patch_y0" in prev_df.columns) and ("patch_x0" in prev_df.columns):
                    for _, row in prev_df.iterrows():
                        y = row.get("patch_y0", None)
                        x = row.get("patch_x0", None)
                        if pd.notna(y) and pd.notna(x):
                            already_done_coords.add((int(y), int(x)))
                results = prev_df.to_dict(orient="records")
                print(f"Loaded {len(results)} previous rows; {len(already_done_coords)} with patch coords")
            except Exception as e:
                print(f"⚠️ Could not read checkpoint {save_path}: {e}")

    def _append_and_checkpoint(rowdict, y0, x0):
        rowdict = dict(rowdict)
        rowdict.setdefault("matcher", str(matcher_mode))
        rowdict.setdefault("patch_h_px", int(patch_h_px))
        rowdict.setdefault("patch_w_px", int(patch_w_px))
        rowdict.setdefault("patch_y0", int(y0))
        rowdict.setdefault("patch_x0", int(x0))

        for c in BENCH_COLS:
            rowdict.setdefault(c, np.nan)

        results.append(rowdict)

        if save_path:
            df_row = pd.DataFrame([rowdict], columns=BENCH_COLS)
            write_header = not os.path.exists(save_path)
            df_row.to_csv(save_path, mode="a", header=write_header, index=False)

        already_done_coords.add((int(y0), int(x0)))

    # Deterministic seed for per-run randomness
    run_seed = int(random_state if random_state is not None else 0)

    # ---- main loop ----
    for (y0, x0) in tqdm(candidates, desc=f"{matcher_mode} | {patch_h_px}x{patch_w_px} px patches"):
        y0 = int(y0); x0 = int(x0)
        if (y0, x0) in already_done_coords:
            continue

        ss = np.random.SeedSequence([int(run_seed), int(patch_h_px), int(patch_w_px), int(y0), int(x0), 999])
        patch_seed = int(ss.generate_state(1, dtype=np.uint32)[0])

        patch_start_t = time.perf_counter()

        y1 = y0 + patch_h_px
        x1 = x0 + patch_w_px
        if y1 > H or x1 > W:
            continue

        patch_img = img_full[y0:y1, x0:x1]

        mask_patch = (
            (df_full["centroid_y_px"] >= y0)
            & (df_full["centroid_y_px"] < y1)
            & (df_full["centroid_x_px"] >= x0)
            & (df_full["centroid_x_px"] < x1)
        )
        df_patch = df_full.loc[mask_patch].copy()
        n_nuc = int(len(df_patch))

        if n_nuc < 3:
            patch_total_t = time.perf_counter() - patch_start_t
            _append_and_checkpoint(dict(
                n_nuclei=n_nuc,
                success=False,
                frac_inliers=0.0,
                mean_error_um=None,
                trans_error_px=None,
                ssim=None,
                time_matcher_s=0.0,
                time_total_s=float(patch_total_t),
            ), y0, x0)
            continue

        # local coords (µm)
        y_local_px = df_patch["centroid_y_px"].to_numpy(dtype=float, copy=False) - float(y0)
        x_local_px = df_patch["centroid_x_px"].to_numpy(dtype=float, copy=False) - float(x0)
        centroids_crop_um = np.column_stack([y_local_px * float(PIXEL_SIZE_UM),
                                             x_local_px * float(PIXEL_SIZE_UM)])

        features_crop = None
        if ("feature_vector" in df_patch.columns) and (len(df_patch) > 0) and (matcher_mode in ("graph", "adaptive")):
            try:
                features_crop = np.stack(df_patch["feature_vector"].to_numpy())
            except Exception:
                features_crop = None

        # ---- call matcher ----
        t0_match = time.perf_counter()

        if matcher_mode == "adaptive":
            controller_kwargs = _sanitize_controller_kwargs(matcher_kwargs)
            controller_kwargs["base_seed"] = int(patch_seed)
            controller_kwargs.setdefault("stop_on_success", True)
            controller_kwargs.setdefault("store_full_out", False)

            out, _hist = run_adaptive_nucleisky(
                centroids_crop_um=centroids_crop_um,
                centroids_full_um=centroids_full_um,
                img_full=img_full,
                img_crop=patch_img,
                pixel_size_full_um=float(PIXEL_SIZE_UM),
                pixel_size_crop_um=float(PIXEL_SIZE_UM),

                # --- FIXED: ADDED BACK ---
                ij_percentile_normalize=ij_percentile_normalize,

                features_crop=features_crop,
                features_full=features_full,
                df_full=df_full,
                df_crop=df_patch,
                matcher_config=matcher_config,
                save_dir=None,
                save_prefix="bench_adaptive",
                **controller_kwargs,
            )
        else:
            bench_overrides = _benchmark_runtime_overrides(matcher_mode, matcher_kwargs, patch_seed)
            out = NucleiSky(
                centroids_crop_um=centroids_crop_um,
                centroids_full_um=centroids_full_um,
                img_full=img_full,
                img_crop=patch_img,
                pixel_size_full_um=float(PIXEL_SIZE_UM),
                pixel_size_crop_um=float(PIXEL_SIZE_UM),
                matcher=str(matcher_mode),

                # --- FIXED: ADDED BACK ---
                ij_percentile_normalize=ij_percentile_normalize,

                features_crop=(features_crop if matcher_mode == "graph" else None),
                features_full=(features_full if matcher_mode == "graph" else None),
                df_full=df_full,
                df_crop=df_patch,
                matcher_config=matcher_config,
                matcher_kwargs=bench_overrides,
                save_dir=None,
                save_prefix="bench",
            )

        time_matcher_s = time.perf_counter() - t0_match

        if out.get("best_t", None) is None:
            patch_total_t = time.perf_counter() - patch_start_t
            frac_inliers, mean_error_um = _extract_quality_safe(out)
            _append_and_checkpoint(dict(
                n_nuclei=n_nuc,
                success=False,
                frac_inliers=float(frac_inliers),
                mean_error_um=mean_error_um,
                trans_error_px=None,
                ssim=None,
                time_matcher_s=float(time_matcher_s),
                time_total_s=float(patch_total_t),
            ), y0, x0)
            continue

        best_t = np.asarray(out["best_t"], float).reshape(2,)
        q = out.get("match_quality", {}) or {}

        success_flag = bool(out.get("success", False))
        try:
            frac_inliers = float(q.get("frac_inliers", np.nan))
        except Exception:
            frac_inliers = np.nan
        mean_error_um = q.get("mean_error_um", None)

        true_topleft_um = np.array([y0 * float(PIXEL_SIZE_UM), x0 * float(PIXEL_SIZE_UM)], dtype=float)
        trans_error_px = float(np.linalg.norm(best_t - true_topleft_um) / float(PIXEL_SIZE_UM))

        ssim_val = None
        if compute_ssim:
            y_pred0 = int(round(best_t[0] / float(PIXEL_SIZE_UM)))
            x_pred0 = int(round(best_t[1] / float(PIXEL_SIZE_UM)))
            if (0 <= y_pred0 <= H - patch_h_px) and (0 <= x_pred0 <= W - patch_w_px):
                pred_roi = img_full[y_pred0:y_pred0 + patch_h_px, x_pred0:x_pred0 + patch_w_px]
                # Still using ij_percentile_normalize for SSIM
                patch_norm = ij_percentile_normalize(patch_img)
                pred_norm  = ij_percentile_normalize(pred_roi)
                ssim_val = float(ssim(patch_norm, pred_norm, data_range=1.0))

        patch_total_t = time.perf_counter() - patch_start_t

        _append_and_checkpoint(dict(
            n_nuclei=n_nuc,
            success=bool(success_flag),
            frac_inliers=frac_inliers,
            mean_error_um=mean_error_um,
            trans_error_px=trans_error_px,
            ssim=ssim_val,
            time_matcher_s=float(time_matcher_s),
            time_total_s=float(patch_total_t),
        ), y0, x0)

    return results


def benchmark_patch_sizes_on_current_image_multi(
    img_full,
    df_full,
    PIXEL_SIZE_UM,
    patch_sizes_px=(20, 50, 75, 100),
    patches_per_size=20,
    random_state=0,
    matchers=("graph", "quad", "triangles", "hashing", "adaptive"),

    matcher_config=None,
    matcher_kwargs_map=None,  # can include "adaptive": {controller kwargs...}

    save_path_dir=None,
    restart=False,
    stride_frac=0.5,
    ij_percentile_normalize=None,

    # NEW
    min_nuclei=1,   # recommend 3 if you keep the n_nuc < 3 early-skip in run_single_patch_match_benchmark
):
    """
    Runs all matchers on the SAME candidate patches for each patch size.

    If min_nuclei > 0, we anchor candidate patches on nuclei (via df_full) and
    filter to contain at least min_nuclei nuclei.
    """
    if ij_percentile_normalize is None:
        raise ValueError("Please pass ij_percentile_normalize.")

    if matcher_kwargs_map is None:
        matcher_kwargs_map = {}

    # Patch plan built ONCE per size -> ensures same patches across matchers
    patch_plan = make_benchmark_patch_plan(
        img_full=img_full,
        patch_sizes_px=patch_sizes_px,
        patches_per_size=int(patches_per_size),
        seed=int(random_state),
        stride_frac=float(stride_frac),

        # NEW: enable anchored mode
        df_full=df_full,
        min_nuclei=int(min_nuclei),
    )

    all_results = []

    for size in patch_sizes_px:
        size = int(size)
        candidates = patch_plan[size]
        print(f"\n=== Patch size {size}×{size} px | N candidates = {len(candidates)} ===")

        for matcher in matchers:
            matcher = str(matcher).strip().lower()
            print(f"\n========== Matcher: {matcher} ==========")

            save_path = None
            if save_path_dir:
                os.makedirs(save_path_dir, exist_ok=True)
                save_path = os.path.join(save_path_dir, f"{matcher}_patch{size}px.csv")

            # deterministic per (global seed, size, matcher)
            ss_run = np.random.SeedSequence([int(random_state), int(size), _stable_u32_from_str(matcher)])
            run_seed = int(ss_run.generate_state(1, dtype=np.uint32)[0])

            res = run_single_patch_match_benchmark(
                img_full=img_full,
                df_full=df_full,
                PIXEL_SIZE_UM=float(PIXEL_SIZE_UM),
                patch_h_px=size,
                patch_w_px=size,
                max_trials=int(patches_per_size),
                random_state=int(run_seed),
                compute_ssim=True,
                matcher=matcher,
                matcher_config=matcher_config,
                matcher_kwargs=matcher_kwargs_map.get(matcher, None),
                save_path=save_path,
                restart=bool(restart),
                candidates=candidates,
                ij_percentile_normalize=ij_percentile_normalize,
            )
            all_results.extend(res)

    df_results = pd.DataFrame(all_results)
    return all_results, df_results



print("Done.")

# 1. Load your images and extract features

In [ ]:
#@title Step 1: Choose the image to benchmark
# ============================================

# ---- Imports ----
import html
import ipywidgets as widgets
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# ----------------------------
# UI Styling & CSS
# ----------------------------
STYLE_CSS = """
<style>
    .ns-card {
        border: 1px solid #E2E8F0;
        border-radius: 8px;
        padding: 16px;
        margin-bottom: 16px;
        background-color: #FFFFFF;
        box-shadow: 0 1px 3px 0 rgb(0 0 0 / 0.1);
    }
    .ns-card-warn {
        border: 1px solid #FECACA; /* Red-200 */
        background-color: #FEF2F2; /* Red-50 */
    }
    .ns-header {
        font-weight: 700;
        font-size: 14px;
        color: #0F172A;
        margin-bottom: 12px;
        text-transform: uppercase;
        letter-spacing: 0.05em;
        border-bottom: 1px solid #F1F5F9;
        padding-bottom: 8px;
    }
    .ns-label {
        font-size: 11px;
        font-weight: 700;
        color: #64748B;
        margin-top: 8px;
        margin-bottom: 4px;
        text-transform: uppercase;
    }
</style>
"""

# ----------------------------
# Utilities
# ----------------------------
def _status_ok(lines):
    safe = "<br>".join(html.escape(str(x)) for x in lines)
    return f"""
    <div style='font-size:13px; color:#065F46; background:#ECFDF5;
    border:1px solid #A7F3D0; padding:10px; border-radius:6px; line-height:1.45;'>
    {safe}</div>"""

def _status_err(lines, title="Action required"):
    safe = "<br>".join(html.escape(str(x)) for x in lines)
    return f"""
    <div style="border:1px solid #FECACA; background:#FEF2F2; border-radius:6px; padding:10px; margin:8px 0;">
      <div style="font-weight:700; color:#991B1B; font-size:13px; margin-bottom:4px;">{html.escape(title)}</div>
      <div style="font-size:12px; color:#7F1D1D; line-height:1.35;">{safe}</div>
    </div>
    """

def require_positive_float(v, label):
    v = float(v)
    if not np.isfinite(v) or v <= 0:
        raise ValueError(f"{label} must be a positive number (µm/px).")
    return v

def _desc_width(*ws, w="90px"):
    for x in ws:
        try: x.style.description_width = w
        except: pass

# ------------------------------------------------------------
# State & Globals
# ------------------------------------------------------------
state = dict(
    confirmed_full=False,
    last_full_path="",
    need_full=False,
)

# Prefill
_last_full = float(globals().get("_NUCLEISKY_LAST_MANUAL_FULL", 1.0))
_default_out_dir = str(globals().get("RESULT_DIR", ""))

# ----------------------------
# WIDGETS CONSTRUCTION
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)

# --- Header ---
title_html = widgets.HTML(
    "<div style='font-weight:900; font-size:18px; color:#0F172A; margin:0;'>Choose Image</div>"
    "<div style='font-size:13px; color:#64748B;'>Select your full reference image and output location.</div>"
)

# --- Card 1: Main Inputs ---
# Set a default value that actually exists if possible, or leave blank to force user input
img_path = widgets.Text(
    value='/content/gdrive/MyDrive/Work/manuscript/Ongoing Projects/NucleiSky/Datasets/microfluidic channel fixed/Image1.tif',
    description='Full Path', placeholder='/path/to/large_image.tif', layout=widgets.Layout(width='98%')
)

out_dir_input = widgets.Text(
    value=_default_out_dir,
    description='Output Dir',
    placeholder='(Optional) Leave empty for auto-timestamp',
    layout=widgets.Layout(width='98%')
)

input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>1. Configuration</div>"),
    img_path,
    widgets.HTML("<div style='height:8px'></div>"),
    widgets.HTML("<div class='ns-label'>Results Directory</div>"),
    out_dir_input,
    widgets.HTML("<div style='font-size:11px; color:#94A3B8; margin-top:2px;'>If left blank, a new timestamped folder will be created automatically.</div>")
])
input_card.add_class("ns-card")

# --- Card 2: Manual Metadata (Warning Card) ---
#
manual_msg = widgets.HTML("")
manual_full = widgets.FloatText(value=_last_full, description='Full µm/px')

manual_card = widgets.VBox([
    widgets.HTML("<div class='ns-header' style='color:#991B1B; border-color:#FECACA;'>Missing Metadata</div>"),
    manual_msg,
    widgets.HBox([manual_full], layout=widgets.Layout(width="100%", margin="5px 0")),
])
manual_card.add_class("ns-card")
manual_card.add_class("ns-card-warn")
manual_card.layout.display = "none"

# --- Outputs ---
status_html = widgets.HTML("")
run_button = widgets.Button(description=' Load & Process', button_style='primary', icon='play', layout=widgets.Layout(width='100%', height='40px'))
fig_out = widgets.Output(layout=widgets.Layout(max_height="500px", overflow="auto"))

# Styling widths
_desc_width(img_path, out_dir_input, w="80px")
_desc_width(manual_full, w="90px")

# ----------------------------
# Logic & Sync
# ----------------------------
def _sync_ui():
    show_manual = state["need_full"]
    manual_card.layout.display = "flex" if show_manual else "none"
    manual_full.layout.visibility = "visible" if state["need_full"] else "hidden"

def _on_paths_changed(_=None):
    fp = img_path.value.strip()

    if fp != state["last_full_path"]:
        state["confirmed_full"] = False
        state["last_full_path"] = fp

    state["need_full"] = False
    manual_msg.value = ""
    _sync_ui()

img_path.observe(_on_paths_changed, names="value")

# ----------------------------
# Main Handler
# ----------------------------
def on_run_clicked(b):
    # Use output widget context immediately to capture print statements
    with fig_out:
        clear_output(wait=True)
        print("Starting processing...") # Immediate feedback

        try:
            global img_full, PIXEL_SIZE_UM, RESULT_DIR
            global _NUCLEISKY_LAST_MANUAL_FULL

            status_html.value = "<div style='color:#2563EB; font-weight:600;'>Processing...</div>"

            _on_paths_changed()
            # 1. Get and clean the path
            full_path_str = img_path.value.strip()
            full_path = Path(full_path_str)
            custom_out = out_dir_input.value.strip()

            # 2. Debug Print to help you see what the computer sees
            if not full_path.exists():
                print(f"❌ ERROR: Could not find file at: '{full_path_str}'")
                print(f"   Current working directory: {Path.cwd()}")
                raise FileNotFoundError(f"Full image not found at: {full_path_str}")
            else:
                print(f"✅ Found image at: {full_path_str}")

            # Load Image
            # Ensure these functions exist in your global scope!
            if 'load_image' not in globals():
                 raise NameError("Function 'load_image' is not defined. Run the setup cells first.")

            full_raw = load_image(str(full_path))
            img_full_2d = require_2d_image(full_raw, label="Full image")

            # Check Pixel Metadata
            missing = []
            pix_full_meta = get_pixel_size_um_from_tiff(str(full_path))

            if pix_full_meta is None:
                if state["confirmed_full"]:
                    pix_full = require_positive_float(manual_full.value, "Full µm/px")
                    pix_full_src = "manual"
                else:
                    missing.append("Full Image")
                    state["need_full"] = True
            else:
                pix_full = float(pix_full_meta)
                pix_full_src = "metadata"
                state["need_full"] = False
                state["confirmed_full"] = False

            if missing:
                if state["need_full"]: state["confirmed_full"] = True
                manual_msg.value = f"<b>Pixel size not found:</b> {', '.join(missing)}."
                status_html.value = ""
                _sync_ui()
                print("⚠️ Missing metadata. Please fill in the red box that appeared.")
                return

            # Handle Output Directory
            if custom_out:
                p_out = Path(custom_out)
                p_out.mkdir(parents=True, exist_ok=True)
                RESULT_DIR = p_out
                print(f"Using custom output directory: {RESULT_DIR}")
            else:
                # Auto-generate timestamped folder
                RESULT_DIR = make_result_dir(big_image_path=str(full_path), tag="NucleiSky")

            # Ensure RESULT_DIR is a Path object for operations below
            RESULT_DIR = Path(RESULT_DIR)

            _NUCLEISKY_LAST_MANUAL_FULL = float(manual_full.value)
            globals()["_NUCLEISKY_LAST_MANUAL_FULL"] = _NUCLEISKY_LAST_MANUAL_FULL

            state["need_full"] = False
            manual_msg.value = ""
            _sync_ui()

            img_full = img_full_2d
            PIXEL_SIZE_UM = float(pix_full)

            # Save inputs
            save_json(RESULT_DIR / "inputs_metadata.json", {
                "full_image_path": str(full_path),
                "pixel_size_full": PIXEL_SIZE_UM,
                "result_dir": str(RESULT_DIR)
            })

            status_html.value = _status_ok([
                f"<b>Success!</b> Working Directory: {RESULT_DIR.name}",
                f"Path: {str(RESULT_DIR)}",
                f"Full: {PIXEL_SIZE_UM:.4f} µm/px ({pix_full_src})",
            ])

            # Display Image
            fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=100)
            imshow_safe(ax, img_full_2d, title="Full Image", max_dim=2500)
            plt.tight_layout()
            plt.show()

        except Exception as e:
            import traceback
            traceback.print_exc()
            status_html.value = _status_err([str(e)], title="Error during processing")

run_button.on_click(on_run_clicked)

# ----------------------------
# Final Display
# ----------------------------
ui = widgets.VBox([
    style_html,
    title_html,
    input_card,
    manual_card,
    widgets.HBox([status_html], layout=widgets.Layout(margin="10px 0")),
    run_button,
    fig_out
], layout=widgets.Layout(width="90%", margin="0 auto"))

_sync_ui()
display(ui)

In [ ]:
#@title Step 2: Segmentation and Feature Extraction

# ---- Imports ----
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import numpy as np
import traceback
import matplotlib.pyplot as plt

# ----------------------------
# UI Styling
# ----------------------------
STYLE_CSS = """
<style>
    .ns-card {
        border: 1px solid #E2E8F0; border-radius: 8px; padding: 16px;
        margin-bottom: 16px; background-color: #FFFFFF;
        box-shadow: 0 1px 3px 0 rgb(0 0 0 / 0.1);
    }
    .ns-header {
        font-weight: 700; font-size: 14px; color: #0F172A;
        margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.05em;
        border-bottom: 1px solid #F1F5F9; padding-bottom: 8px;
    }
    .ns-label { font-size: 12px; font-weight: 600; color: #475569; margin-bottom: 4px; }
    .ns-param-row { display: flex; gap: 12px; align-items: center; margin-bottom: 8px; }
</style>
"""
display(widgets.HTML(STYLE_CSS))

# ----------------------------
# WIDGETS
# ----------------------------

# --- Card 1: Source Selection ---
source_mode = widgets.ToggleButtons(
    options=[('Run New Segmentation', 'new'), ('Load Existing Mask', 'existing')],
    value='new',
    description='',
    button_style='',
    tooltips=['Run algorithms like Cellpose or InstanSeg', 'Load pre-computed binary TIFF mask'],
    layout=widgets.Layout(width='auto')
)

# --- Path Inputs (For Existing Masks) ---
mask_path_full = widgets.Text(description="Mask Path", placeholder="/path/to/mask_full.tif", layout=widgets.Layout(width="98%"))

mask_input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Mask Path</div>"),
    mask_path_full
])
mask_input_card.layout.display = "none" # Hidden by default

# --- Card 2: Segmentation Method ---
seg_method = widgets.Dropdown(
    options=[
        ("Cellpose (Deep Learning)", "cellpose"),
        ("InstanSeg (Fast Deep Learning)", "instanseg"),
        ("Auto-threshold (Classic CV)", "threshold"),
    ],
    value="cellpose",
    layout=widgets.Layout(width="98%")
)

method_desc = widgets.HTML("") # Dynamic description

# ---- InstanSeg Specifics ----
inst_model = widgets.Dropdown(options=["brightfield_nuclei", "fluorescence_nuclei_and_cells"], value="brightfield_nuclei", description="Model")
inst_target = widgets.Dropdown(options=["nuclei", "cells"], value="nuclei", description="Target")

# Advanced InstanSeg
inst_mode = widgets.Dropdown(options=[("Auto (small/medium)", "auto"), ("Small image", "small"), ("Medium (tiled)", "medium")], value="auto", description="Mode")
inst_clean = widgets.Checkbox(value=True, description="Cleanup Fragments")
inst_px_ovr = widgets.Checkbox(value=False, description="Override µm/px")
inst_px_val = widgets.FloatText(value=0.5, description="Value", layout=widgets.Layout(width="150px"), disabled=True)
inst_verb = widgets.IntSlider(value=0, min=0, max=2, description="Verbosity")

inst_adv_ui = widgets.VBox([
    inst_mode,
    widgets.HBox([inst_px_ovr, inst_px_val]),
    inst_clean,
    inst_verb
])
inst_accord = widgets.Accordion(children=[inst_adv_ui], titles=('Advanced InstanSeg Settings',))

inst_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Model Configuration</div>"),
    widgets.HBox([inst_model, inst_target], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<div style='height:8px'></div>"),
    inst_accord
])

# ---- Threshold Specifics ----
thr_method = widgets.Dropdown(options=["otsu", "li", "yen", "triangle", "isodata"], value="otsu", description="Algorithm")
thr_channel = widgets.IntText(value=0, description="Channel Index", layout=widgets.Layout(width="180px"))

# Advanced Threshold
thr_sigma = widgets.FloatSlider(value=1.0, min=0, max=5, step=0.25, description="Blur Sigma")
thr_min_obj = widgets.IntText(value=80, description="Min Area (px)")
thr_watershed = widgets.Checkbox(value=True, description="Watershed Split")
thr_peak = widgets.IntText(value=5, description="Peak Dist")

thr_adv_ui = widgets.VBox([
    thr_sigma,
    widgets.HBox([thr_min_obj, thr_watershed, thr_peak])
])
thr_accord = widgets.Accordion(children=[thr_adv_ui], titles=('Advanced CV Parameters',))

thr_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Thresholding Configuration</div>"),
    widgets.HBox([thr_method, thr_channel], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<div style='height:8px'></div>"),
    thr_accord
])

# Container for all method panels
method_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Segmentation Strategy</div>"),
    seg_method,
    widgets.HTML("<div style='height:12px'></div>"),
    method_desc,
    inst_panel,
    thr_panel
])
method_card.add_class("ns-card")

# --- Card 3: Execution ---
seg_btn = widgets.Button(
    description=" Run Segmentation",
    button_style="primary",
    icon="cogs",
    layout=widgets.Layout(width="100%", height="45px")
)

seg_out = widgets.Output(layout=widgets.Layout(max_height="600px", overflow="auto"))

# ----------------------------
# LOGIC & BINDINGS
# ----------------------------

def _update_ui_state(_=None):
    is_existing = (source_mode.value == 'existing')

    # Toggle Cards
    mask_input_card.layout.display = "flex" if is_existing else "none"
    method_card.layout.display = "none" if is_existing else "flex"

    # Toggle Method Panels
    m = seg_method.value
    inst_panel.layout.display = "flex" if m == "instanseg" else "none"
    thr_panel.layout.display = "flex" if m == "threshold" else "none"

    # Dynamic Description & Image Tags
    if m == "cellpose":
        method_desc.value = "<i>Standard deep learning model for cellular segmentation. Good general purpose choice.</i>"
    elif m == "instanseg":
        method_desc.value = "<i>High-speed pytorch implementation. Best for large datasets.</i>"
    elif m == "threshold":
        method_desc.value = "<i>Classical computer vision. Fast, but requires high contrast and clean images.</i>"

    # Button Text
    seg_btn.description = " Load Mask & Extract Features" if is_existing else " Run Segmentation & Extraction"

# InstanSeg Override logic
def _toggle_inst_px(_=None):
    inst_px_val.disabled = not inst_px_ovr.value
inst_px_ovr.observe(_toggle_inst_px, names="value")

source_mode.observe(_update_ui_state, names="value")
seg_method.observe(_update_ui_state, names="value")

# Initialize
_update_ui_state()

# ----------------------------
# EXECUTION HANDLER
# ----------------------------
def on_segment_click(_):
    # Mapping global variables to new UI
    global df_full, masks_full
    global img_full_seg
    global PIXEL_SIZE_UM_FULL_SEG

    seg_out.clear_output(wait=True)

    with seg_out:
        try:
            # 1. Pre-flight Checks
            required = ("img_full", "PIXEL_SIZE_UM")
            missing = [k for k in required if k not in globals()]
            if missing: raise RuntimeError(f"Missing Step 1 globals: {missing}")

            # 2. Get Data
            img_full_orig = require_2d_image(np.asarray(img_full), label="Full Image")
            pix_full_orig = float(globals()["PIXEL_SIZE_UM"])

            # 3. Setup Directories
            RESULT_DIR_LOCAL = Path(globals().get("RESULT_DIR", make_result_dir(tag="NucleiSky")))
            SEG_DIR = RESULT_DIR_LOCAL / "segmentation"
            SEG_DIR.mkdir(parents=True, exist_ok=True)

            # 4. Processing Mode Logic
            use_masks = (source_mode.value == 'existing')

            # We process directly on the full image resolution
            # We assume segmentation functions handle internal resizing if pixel_size_um is passed
            img_full_seg = img_full_orig
            PIXEL_SIZE_UM_FULL_SEG = pix_full_orig

            # Factor is 1.0 because we are using the original image directly
            RESCALE_FACTOR_FULL = 1.0

            if use_masks:
                print(f"📂 Loading existing mask...")

                if not mask_path_full.value.strip(): raise ValueError("Mask path required")

                masks_full = require_2d_label_mask(load_image(mask_path_full.value.strip()), label="Full Mask", expected_shape=img_full_orig.shape)

            else:
                # PERFORM SEGMENTATION
                print(f"🧠 Running {seg_method.value}...")

                # Build Settings Dictionary
                settings = {"method": seg_method.value}

                # Override logic
                pix_dispatch_full = float(PIXEL_SIZE_UM_FULL_SEG)

                if seg_method.value == "instanseg":
                    if inst_px_ovr.value:
                        pix_dispatch_full = float(inst_px_val.value)

                    settings["instanseg"] = {
                        "model_name": inst_model.value, "target": inst_target.value,
                        "mode": inst_mode.value, "verbosity": inst_verb.value,
                        "pixel_size_um": float(pix_dispatch_full),
                        "cleanup_fragments": inst_clean.value
                    }
                elif seg_method.value == "threshold":
                    #
                    settings["threshold"] = {
                        "threshold_method": thr_method.value, "channel": thr_channel.value,
                        "gaussian_sigma": thr_sigma.value, "min_object_size": thr_min_obj.value,
                        "do_watershed": thr_watershed.value, "peak_min_distance": thr_peak.value
                    }

                masks_full = segment_nuclei_dispatch(img_full_seg, seg_method.value, pix_dispatch_full, settings)

            # 5. Feature Extraction (Common)
            print("📊 Extracting morphology features...")
            df_full = extract_nuclear_features(masks_full, None, float(PIXEL_SIZE_UM_FULL_SEG), k_neighbors=10)

            # Map back to original scale (even if scale is 1.0, this ensures the correct columns 'centroid_x_px', etc. are created)
            if 'add_centroids_orig_px_columns' in globals():
                df_full = add_centroids_orig_px_columns(df_full, float(RESCALE_FACTOR_FULL))

            # 6. Saving Data
            print(f"✅ Extracted: {len(df_full)} nuclei")
            df_full.to_csv(SEG_DIR / "features_full.csv", index=False)

            # --- Save Masks with ZLIB Compression ---

            print("💾 Saving compressed segmentation mask to disk...")
            imwrite(
                SEG_DIR / "mask_full.tif",
                masks_full.astype(np.int32),
                compression="zlib"
            )

            # 7. Visualization
            fig, ax = plt.subplots(1, 1, figsize=(8, 8))

            # --- Safe Visualization ---
            disp_full = imshow_safe(ax, img_full_orig, title=f"Segmentation ({len(df_full)})")

            # Calculate coordinate scale (Display Shape / Original Shape)
            scale_y_f, scale_x_f = disp_full.shape[0] / img_full_orig.shape[0], disp_full.shape[1] / img_full_orig.shape[1]

            # Plot centroids if available
            if 'centroids_from_df' in globals():
                cent_f = centroids_from_df(df_full, use_um=False, use_orig_px=True)
                if len(cent_f):
                    # Downsample points to avoid UI crashes
                    cent_f_safe = downsample_points_for_display(cent_f)
                    # Apply scale to align points with the downsampled image axes
                    ax.scatter(cent_f_safe[:,1] * scale_x_f, cent_f_safe[:,0] * scale_y_f, s=1, c='r', alpha=0.5)

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print("❌ Error:")
            traceback.print_exc()

seg_btn.on_click(on_segment_click)

# ----------------------------
# UI Layout
# ----------------------------
ui = widgets.VBox([
    widgets.HTML("<div class='ns-header'>1. Source</div>"),
    source_mode,
    widgets.HTML("<div style='height:16px'></div>"),
    mask_input_card,
    method_card,
    widgets.HTML("<div style='height:16px'></div>"),
    seg_btn,
    seg_out
], layout=widgets.Layout(width="90%"))

display(ui)

# 2. Benchmark all matches vs one another using patches

In [ ]:
#@title Run the Benchmark

# ---------------------------------------------------------
# DYNAMIC PATH SETUP
# ---------------------------------------------------------
# Alternative 1: Use the RESULT_DIR from Step 1
if "RESULT_DIR" in globals() and RESULT_DIR:
    base_output_dir = Path(RESULT_DIR)
else:
    # Alternative 2: Fallback to local directory if pipeline wasn't run linearly
    base_output_dir = Path("benchmark_results")

# Create a dedicated subfolder for benchmarks to keep things clean
output_dir = base_output_dir / "benchmarking"
output_dir.mkdir(parents=True, exist_ok=True)

# Try to get the original filename from the metadata saved in Step 1
base_name = "current_image"
try:
    meta_path = base_output_dir / "inputs_metadata.json"
    if meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)
            if "full_image_path" in meta:
                base_name = Path(meta["full_image_path"]).stem
except Exception:
    pass

print(f"📂 Saving benchmark results to: {output_dir}")
print(f"🏷️  Base filename: {base_name}")

# ---------------------------------------------------------
# PREPARE DATA
# ---------------------------------------------------------
# Use segmentation-scale image + pixel size if available
img_for_bench = globals().get("img_full_seg", globals().get("img_full"))
pix_for_bench = float(globals().get("PIXEL_SIZE_UM_FULL_SEG", globals().get("PIXEL_SIZE_UM")))

if img_for_bench is None:
    raise RuntimeError("Image data not found in globals. Please run Step 1 & 2 first.")

print("\nBenchmark settings:")
print(f"  Image shape: {np.asarray(img_for_bench).shape}")
print(f"  Scale:       {pix_for_bench:.4f} µm/px")

# ---------------------------------------------------------
# RUN BENCHMARK
# ---------------------------------------------------------
all_results, df_results = benchmark_patch_sizes_on_current_image_multi(
    img_full=img_for_bench,
    df_full=df_full,
    PIXEL_SIZE_UM=pix_for_bench,
    # Standard grid sizes
    patch_sizes_px=[50, 75, 100, 128, 192, 256, 320, 384, 448, 512],
    patches_per_size=50,
    random_state=0,
    matchers=("graph", "quad", "triangles", "hashing", "adaptive"),

    matcher_config=None,
    matcher_kwargs_map={
        "adaptive": {
            "mode": "balanced",
            "max_rounds": 2,
            # We put graph first in benchmark to see if it works optimally
            "matcher_order": ["graph", "triangles", "quad", "hashing"],
            "max_total_time_s": None,
        }
    },

    min_nuclei=3,

    # Dynamic checkpoint path
    save_path_dir=str(output_dir / f"{base_name}_checkpoints"),
    restart=False,
    ij_percentile_normalize=ij_percentile_normalize,
)

# ---------------------------------------------------------
# SAVE RESULTS
# ---------------------------------------------------------
csv_path = output_dir / f"{base_name}_benchmark_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"\n✅ Saved final benchmark CSV to:\n   {csv_path}")

## 2.1. Plot Results

In [ ]:
#@title reload result files if needed

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Check if df_results is already in the global environment
if 'df_results' not in globals():
    print("⚠️ 'df_results' not found in environment. Please provide the path to your CSV.")

    # Create the text input widget
    path_input = widgets.Text(
        value='',
        placeholder='e.g., /path/to/your/results.csv',
        description='CSV Path:',
        layout=widgets.Layout(width='50%')
    )

    # Create the load button
    load_button = widgets.Button(
        description='Load CSV',
        button_style='info',
        tooltip='Click to load the dataframe'
    )

    # Create an output area for success/error messages
    output_area = widgets.Output()

    def on_button_clicked(b):
        with output_area:
            output_area.clear_output()
            csv_path = path_input.value.strip()
            if not csv_path:
                print("❌ Please enter a valid path.")
                return

            try:
                # Declare global so it becomes available to the rest of the notebook
                global df_results
                df_results = pd.read_csv(csv_path)
                print(f"✅ Successfully loaded {len(df_results)} rows from '{csv_path}' into df_results!")
            except Exception as e:
                print(f"❌ Error loading file: {e}")

    load_button.on_click(on_button_clicked)

    # Display the widgets together
    display(widgets.HBox([path_input, load_button]), output_area)

else:
    print("✅ 'df_results' is already in the environment. Skipping file load.")

In [ ]:
#@title Plots: success probability vs nuclei count, patch size and speed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.optimize import curve_fit
from pathlib import Path
from IPython.display import display, Markdown

# ==========================================
# 0. Plotting Configuration for Editable PDFs
# ==========================================

# Export fonts as editable text objects in Illustrator/Inkscape
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"

# ==========================================
# 1. Configuration & Data Prep
# ==========================================

TARGET_PROB = 0.90
FRAC_INLIERS_THRESH = 0.80
SSIM_THRESH = 0.80
ERROR_PX_THRESH = 20

# Output directory for plots and summary tables
output_dir = Path("pdf_plots")
output_dir.mkdir(parents=True, exist_ok=True)


def normalise_matcher_name(matcher):
    """
    Normalise matcher names so colours remain consistent.
    """
    matcher = str(matcher).strip().lower()

    aliases = {
        "triangle": "triangles",
        "tri": "triangles",
        "geo_hashing": "hashing",
        "geometric_hashing": "hashing",
        "geometric hashing": "hashing",
        "quad_matcher": "quad",
        "triangle_matcher": "triangles",
        "graph_matcher": "graph",
        "hashing_matcher": "hashing",
    }

    return aliases.get(matcher, matcher)


def sanitize_and_score(df_results):
    """
    Prepare the standard NucleiSky benchmark dataframe.

    Success definition for the standard benchmark:
        matcher success flag == True
        frac_inliers >= FRAC_INLIERS_THRESH
        SSIM >= SSIM_THRESH
        translation error < ERROR_PX_THRESH

    This is appropriate for the standard benchmark where the clean point sets
    are used. Do not use this success definition for the segmentation-error
    benchmark, where SSIM-only success is preferred.
    """
    df = pd.DataFrame(df_results).copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x) for x in t if str(x) != ""]).strip()
            for t in df.columns
        ]

    df.columns = [str(c).strip() for c in df.columns]

    required_cols = ["matcher", "n_nuclei", "success", "frac_inliers", "ssim"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"df_results is missing required columns: {missing}")

    df["matcher"] = df["matcher"].map(normalise_matcher_name)

    numeric_cols = [
        "n_nuclei",
        "patch_h_px",
        "patch_w_px",
        "frac_inliers",
        "ssim",
        "time_total_s",
        "trans_error_px",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["n_nuclei"] = df["n_nuclei"].fillna(0).astype(int)
    df["success"] = df["success"].fillna(False).astype(bool)

    if "trans_error_px" not in df.columns:
        df["trans_error_px"] = 0.0

    df["is_success"] = (
        (df["success"] == True) &
        (df["frac_inliers"] >= FRAC_INLIERS_THRESH) &
        (df["ssim"] >= SSIM_THRESH) &
        (df["trans_error_px"] < ERROR_PX_THRESH)
    )

    return df


def wilson_interval(k, n, z=1.96):
    """
    95% Wilson confidence interval for a binomial proportion.
    """
    if n <= 0:
        return np.nan, np.nan

    p_hat = k / n
    denom = 1 + z**2 / n
    centre = (p_hat + z**2 / (2 * n)) / denom
    half_width = z * np.sqrt(
        p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)
    ) / denom

    return max(0.0, centre - half_width), min(1.0, centre + half_width)


# ==========================================
# 2. Consistent matcher colour map
# ==========================================

PREFERRED_MATCHER_ORDER = [
    "quad",
    "triangles",
    "graph",
    "hashing",
    "adaptive",
]

# Fixed, colourblind-friendly colours.
# These remain stable even if not all matchers are present in a dataset.
FIXED_MATCHER_COLORS = {
    "hashing": "#009E73",    # green
    "graph": "#E69F00",      # orange
    "quad": "#D62728",       # red
    "triangles": "#CC79A7",  # purple
    "adaptive": "#0072B2",   # blue
}


def get_matcher_order(df):
    """
    Return a stable matcher order for plotting.
    Known matchers follow PREFERRED_MATCHER_ORDER.
    Unknown matchers are appended alphabetically.
    """
    present = sorted(df["matcher"].dropna().map(normalise_matcher_name).unique())

    ordered = [m for m in PREFERRED_MATCHER_ORDER if m in present]
    ordered += [m for m in present if m not in ordered]

    return ordered


def make_matcher_color_map(matchers):
    """
    Build a colour map that is stable for known matchers and robust to new ones.
    """
    color_map = {}

    for matcher in matchers:
        if matcher in FIXED_MATCHER_COLORS:
            color_map[matcher] = FIXED_MATCHER_COLORS[matcher]

    unknown = [m for m in matchers if m not in color_map]
    if unknown:
        extra_palette = sns.color_palette("colorblind", n_colors=len(unknown))
        for matcher, color in zip(unknown, extra_palette):
            color_map[matcher] = color

    return color_map


# ==========================================
# 3. Mathematical modelling
# ==========================================

def logistic_log_n(log_n, k, log_n0):
    """
    Logistic model in log10(nuclei) space.

    This is preferable when nuclei counts span a wide range but the important
    transition occurs at low nuclei numbers.
    """
    return 1.0 / (1.0 + np.exp(-k * (log_n - log_n0)))


def nuclei_for_target_from_logistic(k, log_n0, target=0.90):
    """
    Return the nuclei count at which the fitted logistic curve reaches target.
    """
    if k <= 0 or target <= 0 or target >= 1:
        return np.nan

    log_n_target = log_n0 - np.log((1.0 / target) - 1.0) / k
    return 10 ** log_n_target


def make_log_nuclei_bins(n_values):
    """
    Make biology-friendly bins for nuclei counts.

    The bins are fine at low nuclei counts and wider at high nuclei counts.
    This keeps the low-nuclei regime readable while still covering dense fields.
    """
    n_values = np.asarray(n_values, dtype=float)
    n_values = n_values[np.isfinite(n_values) & (n_values > 0)]

    if len(n_values) == 0:
        raise ValueError("No positive nuclei counts available.")

    max_n = int(np.nanmax(n_values))

    # Fine resolution in the critical low-nuclei regime.
    low_edges = np.array([1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50])

    if max_n <= 50:
        edges = np.arange(1, max_n + 2, 1)
    else:
        high_edges = np.unique(
            np.round(
                np.logspace(np.log10(50), np.log10(max_n + 1), 12)
            ).astype(int)
        )
        edges = np.unique(np.concatenate([low_edges, high_edges]))
        edges = edges[(edges >= 1) & (edges <= max_n + 1)]

    if len(edges) < 3:
        edges = np.array([1, max_n + 1])

    return edges


def empirical_threshold_from_bins(binned, target=0.90, require_sustained=True):
    """
    Estimate the empirical nuclei threshold from binned observed probabilities.

    If require_sustained=True, returns the first bin after which all later bins
    remain at or above the target. This is conservative and avoids calling a
    single noisy high bin a threshold.
    """
    b = binned.sort_values("median_nuclei").copy()

    if len(b) == 0:
        return np.nan

    y = b["success_probability"].to_numpy(dtype=float)
    x = b["median_nuclei"].to_numpy(dtype=float)

    if require_sustained:
        for i in range(len(b)):
            later = y[i:]
            if np.all(later >= target):
                return float(x[i])
        return np.nan

    above = b[y >= target]
    if len(above) == 0:
        return np.nan

    return float(above["median_nuclei"].iloc[0])


# ==========================================
# 4. Plotting functions
# ==========================================

def plot_success_and_calc_min_n_best(
    df,
    target=0.90,
    save_path=None,
    save_table_path=None,
    require_sustained_empirical=True,
):
    """
    Best-practice plot for localisation success probability versus nuclei count.

    Main display:
      - observed success probability in nuclei-count bins
      - Wilson 95% confidence intervals
      - log-scaled x-axis
      - logistic fit in log10(nuclei) space as a smooth visual summary

    Summary table:
      - empirical nuclei threshold for target success
      - fitted nuclei threshold for target success
    """
    df = pd.DataFrame(df).copy()

    required = ["matcher", "n_nuclei", "is_success"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["matcher"] = df["matcher"].map(normalise_matcher_name)
    df["n_nuclei"] = pd.to_numeric(df["n_nuclei"], errors="coerce")
    df["is_success"] = df["is_success"].fillna(False).astype(bool)

    # A log x-axis cannot display zero.
    df = df[np.isfinite(df["n_nuclei"]) & (df["n_nuclei"] > 0)].copy()

    if df.empty:
        raise ValueError("No rows with n_nuclei > 0 available for plotting.")

    fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=140)

    summary_rows = []

    for matcher in MATCHER_ORDER:
        if matcher not in set(df["matcher"].dropna().unique()):
            continue

        grp = df[df["matcher"] == matcher].copy()
        color = MATCHER_COLOR_MAP[matcher]

        bin_edges = make_log_nuclei_bins(grp["n_nuclei"].values)

        grp["nuclei_bin"] = pd.cut(
            grp["n_nuclei"],
            bins=bin_edges,
            include_lowest=True,
            duplicates="drop",
        )

        binned = (
            grp
            .groupby("nuclei_bin", observed=True)
            .agg(
                n_trials=("is_success", "size"),
                n_success=("is_success", "sum"),
                median_nuclei=("n_nuclei", "median"),
                mean_nuclei=("n_nuclei", "mean"),
                min_nuclei=("n_nuclei", "min"),
                max_nuclei=("n_nuclei", "max"),
                success_probability=("is_success", "mean"),
            )
            .reset_index()
            .dropna(subset=["median_nuclei", "success_probability"])
        )

        if len(binned) == 0:
            summary_rows.append({
                "matcher": matcher,
                "n_trials": len(grp),
                f"empirical_min_nuclei_{int(target * 100)}": np.nan,
                f"fit_min_nuclei_{int(target * 100)}": np.nan,
                "fit_status": "no binned data",
            })
            continue

        ci = binned.apply(
            lambda r: wilson_interval(r["n_success"], r["n_trials"]),
            axis=1,
            result_type="expand",
        )

        binned["ci_low"] = ci[0]
        binned["ci_high"] = ci[1]

        # Observed binned success probability
        ax.plot(
            binned["median_nuclei"],
            binned["success_probability"],
            "o",
            color=color,
            alpha=0.90,
            markersize=5,
            label=f"{matcher} observed",
        )

        # Wilson confidence interval
        ax.fill_between(
            binned["median_nuclei"].to_numpy(dtype=float),
            binned["ci_low"].to_numpy(dtype=float),
            binned["ci_high"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=0,
        )

        # Empirical threshold from observed binned probabilities
        empirical_n = empirical_threshold_from_bins(
            binned,
            target=target,
            require_sustained=require_sustained_empirical,
        )

        # Logistic fit in log10(nuclei) space
        fit_n = np.nan
        fit_status = "fit failed"

        try:
            x_raw = grp["n_nuclei"].to_numpy(dtype=float)
            y_raw = grp["is_success"].to_numpy(dtype=float)

            good = np.isfinite(x_raw) & (x_raw > 0) & np.isfinite(y_raw)
            x_raw = x_raw[good]
            y_raw = y_raw[good]

            # Need both successes and failures for an informative logistic fit.
            if len(np.unique(y_raw)) < 2:
                fit_status = "all successes or all failures"
            else:
                x_log = np.log10(x_raw)

                p0 = [4.0, np.nanmedian(x_log)]
                bounds = ([0.01, np.nanmin(x_log)], [50.0, np.nanmax(x_log)])

                popt, _ = curve_fit(
                    logistic_log_n,
                    x_log,
                    y_raw,
                    p0=p0,
                    bounds=bounds,
                    maxfev=20000,
                )

                k, log_n0 = popt
                fit_n = nuclei_for_target_from_logistic(k, log_n0, target=target)

                x_fit = np.logspace(
                    np.log10(np.nanmin(x_raw)),
                    np.log10(np.nanmax(x_raw)),
                    300,
                )
                y_fit = logistic_log_n(np.log10(x_fit), k, log_n0)

                ax.plot(
                    x_fit,
                    y_fit,
                    color=color,
                    linewidth=2.0,
                    alpha=0.85,
                    label=f"{matcher} fit",
                )

                fit_status = "ok"

        except Exception as e:
            fit_status = f"fit failed: {type(e).__name__}"

        summary_rows.append({
            "matcher": matcher,
            "n_trials": len(grp),
            f"empirical_min_nuclei_{int(target * 100)}": empirical_n,
            f"fit_min_nuclei_{int(target * 100)}": fit_n,
            "fit_status": fit_status,
        })

    # Target line
    ax.axhline(
        target,
        color="0.35",
        linestyle="--",
        linewidth=1.0,
        alpha=0.9,
    )

    ax.text(
        0.02,
        target + 0.035,
        f"{int(target * 100)}% success",
        transform=ax.get_yaxis_transform(),
        fontsize=9,
        color="0.3",
    )

    # Axis styling
    ax.set_xscale("log")

    max_n = int(np.nanmax(df["n_nuclei"]))
    min_n = max(1, int(np.nanmin(df["n_nuclei"])))

    ax.set_xlim(max(1, min_n * 0.85), max_n * 1.15)
    ax.set_ylim(-0.04, 1.04)

    candidate_ticks = np.array([
        1, 2, 3, 5, 10, 20, 50,
        100, 200, 500, 1000, 2000, 5000
    ])

    ticks = candidate_ticks[
        (candidate_ticks >= max(1, min_n)) &
        (candidate_ticks <= max_n * 1.15)
    ]

    ax.set_xticks(ticks)
    ax.set_xticklabels([str(int(t)) for t in ticks])

    ax.set_xlabel("Number of nuclei in query field of view")
    ax.set_ylabel("Probability of correct localisation")
    ax.set_title("Localisation success increases with nuclear constellation size")

    ax.grid(True, which="both", alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, format="pdf", bbox_inches="tight", transparent=True)
        print(f"Saved: {save_path}")

    summary = pd.DataFrame(summary_rows)

    sort_col = f"empirical_min_nuclei_{int(target * 100)}"
    if sort_col in summary.columns:
        summary = summary.sort_values(sort_col, na_position="last")

    if save_table_path:
        summary.to_csv(save_table_path, index=False)
        print(f"Saved table: {save_table_path}")

    plt.show()

    return summary


def plot_success_vs_patch_size(df, save_path=None):
    """
    Plot success probability as a function of patch size using Wilson intervals.
    """
    fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=140)

    df = df.copy()
    df["matcher"] = df["matcher"].map(normalise_matcher_name)

    grouped = (
        df
        .groupby(["matcher", "patch_h_px"], observed=True)
        .agg(
            n_patches=("is_success", "count"),
            k_success=("is_success", "sum"),
        )
        .reset_index()
    )

    for matcher in MATCHER_ORDER:
        if matcher not in set(grouped["matcher"].dropna().unique()):
            continue

        g = grouped[grouped["matcher"] == matcher].copy()
        g = g.sort_values("patch_h_px")

        x = g["patch_h_px"].to_numpy(dtype=float)
        y = g["k_success"].to_numpy(dtype=float) / g["n_patches"].to_numpy(dtype=float)

        bounds = [
            wilson_interval(k, n)
            for k, n in zip(g["k_success"], g["n_patches"])
        ]

        lo = np.array([b[0] for b in bounds], dtype=float)
        hi = np.array([b[1] for b in bounds], dtype=float)

        color = MATCHER_COLOR_MAP[matcher]

        ax.plot(
            x,
            y,
            "o-",
            color=color,
            linewidth=2.0,
            markersize=5,
            label=matcher,
        )

        ax.fill_between(
            x,
            lo,
            hi,
            color=color,
            alpha=0.15,
            linewidth=0,
        )

    ax.set_xlabel("Patch size (px)")
    ax.set_ylabel("Probability of correct localisation")
    ax.set_title("Localisation success versus query patch size")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    ax.grid(True, alpha=0.25)
    ax.set_ylim(-0.04, 1.04)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, format="pdf", bbox_inches="tight", transparent=True)
        print(f"Saved: {save_path}")

    plt.show()


def plot_speed_vs_nuclei(df, save_path=None):
    """
    Plot mean runtime as a function of nuclei count.
    Uses log-scaled x-axis if the nuclei-count range is wide.
    """
    fig, ax = plt.subplots(figsize=(7.2, 4.8), dpi=140)

    if "time_total_s" not in df.columns:
        print("time_total_s not found; skipping speed plot.")
        return

    df = df.copy()
    df["matcher"] = df["matcher"].map(normalise_matcher_name)
    df["time_total_s"] = pd.to_numeric(df["time_total_s"], errors="coerce")
    df["n_nuclei"] = pd.to_numeric(df["n_nuclei"], errors="coerce")

    df = df[np.isfinite(df["n_nuclei"]) & (df["n_nuclei"] > 0)].copy()

    for matcher in MATCHER_ORDER:
        if matcher not in set(df["matcher"].dropna().unique()):
            continue

        g = df[df["matcher"] == matcher].copy()
        color = MATCHER_COLOR_MAP[matcher]

        bin_edges = make_log_nuclei_bins(g["n_nuclei"].values)

        g["nuclei_bin"] = pd.cut(
            g["n_nuclei"],
            bins=bin_edges,
            include_lowest=True,
            duplicates="drop",
        )

        g_sorted = (
            g
            .groupby("nuclei_bin", observed=True)
            .agg(
                median_nuclei=("n_nuclei", "median"),
                mean_time_s=("time_total_s", "mean"),
            )
            .reset_index()
            .dropna(subset=["median_nuclei", "mean_time_s"])
            .sort_values("median_nuclei")
        )

        ax.plot(
            g_sorted["median_nuclei"],
            g_sorted["mean_time_s"],
            "o-",
            color=color,
            linewidth=2.0,
            markersize=5,
            label=matcher,
        )

    ax.set_xscale("log")

    ax.set_xlabel("Number of nuclei in query field of view")
    ax.set_ylabel("Mean runtime (s)")
    ax.set_title("Runtime versus nuclear constellation size")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    ax.grid(True, which="both", alpha=0.25)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, format="pdf", bbox_inches="tight", transparent=True)
        print(f"Saved: {save_path}")

    plt.show()


# ==========================================
# 5. Execution Flow
# ==========================================

if "df_results" not in globals():
    raise RuntimeError(
        "df_results is not defined. Load your standard benchmark CSV into df_results first."
    )

df_clean = sanitize_and_score(df_results)

MATCHER_ORDER = get_matcher_order(df_clean)
MATCHER_COLOR_MAP = make_matcher_color_map(MATCHER_ORDER)

print(f"Total evaluated patches: {len(df_clean)}")
print(
    "Success criteria: "
    f"matcher success = True, "
    f"inlier fraction ≥ {FRAC_INLIERS_THRESH}, "
    f"SSIM ≥ {SSIM_THRESH}, "
    f"translation error < {ERROR_PX_THRESH}px"
)

print("\nConsistent matcher colours:")
for matcher in MATCHER_ORDER:
    print(f"  {matcher}: {MATCHER_COLOR_MAP[matcher]}")

display(Markdown(
    "### Probability of correct localisation versus nuclei count\n"
    "The main points show observed success probability in nuclei-count bins. "
    "Shaded regions are Wilson 95% confidence intervals. "
    "Smooth curves are logistic fits in log10(nuclei) space and should be interpreted as visual summaries. "
    "Matcher colours are fixed and reused across all plots."
))

# Plot 1: Probability of correct localisation vs number of nuclei
min_req_table = plot_success_and_calc_min_n_best(
    df_clean,
    target=TARGET_PROB,
    save_path=output_dir / "success_vs_nuclei_logx_empirical_wilson.pdf",
    save_table_path=output_dir / "success_vs_nuclei_thresholds.csv",
)

print(f"\nMinimal nuclei count for {int(TARGET_PROB * 100)}% localisation probability:")
display(
    min_req_table
    .style
    .format({
        f"empirical_min_nuclei_{int(TARGET_PROB * 100)}": "{:.1f}",
        f"fit_min_nuclei_{int(TARGET_PROB * 100)}": "{:.1f}",
    })
    .background_gradient(
        subset=[f"empirical_min_nuclei_{int(TARGET_PROB * 100)}"],
        cmap="Greens_r",
    )
)

# Plot 2: Success vs Patch Size
if "patch_h_px" in df_clean.columns:
    plot_success_vs_patch_size(
        df_clean,
        save_path=output_dir / "success_vs_patch_size.pdf",
    )
else:
    print("patch_h_px not found; skipping success vs patch size plot.")

# Plot 3: Speed vs Nuclei
plot_speed_vs_nuclei(
    df_clean,
    save_path=output_dir / "speed_vs_nuclei_logx.pdf",
)

# 3. Segmentation-error benchmark


In [ ]:
#@title NucleiSky segmentation-error sensitivity: missed nuclei and extra detections, SSIM-only success, resumable
# This benchmark deliberately corrupts centroid sets to mimic segmentation errors.
#
# IMPORTANT:
# - Adaptive mode is intentionally excluded.
# - Matcher-reported success is NOT used as the benchmark success criterion.
# - frac_inliers is NOT used as the benchmark success criterion.
# - Benchmark success is defined ONLY by SSIM between:
#       1. the true query image patch
#       2. the predicted reference crop
#
# Resume behaviour:
# - If previous benchmark data exist in the output folder, the cell reloads them.
# - Completed matcher jobs are skipped.
# - If no previous data exist, the benchmark starts from scratch.
#
# Why:
#   False negatives reduce the maximum possible inlier fraction.
#   False positives change the consensus denominator and can distort internal
#   success flags. Therefore, inlier fraction is recorded only as a diagnostic.

import time
import json
import zlib
import io
import contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim

# ---------------------------------------------------------------------
# 1. User settings
# ---------------------------------------------------------------------

PATCH_SIZE_PX = 512

# Debug run:
# PATCHES_PER_SIZE = 1
# N_REPEATS = 1

# Benchmark run:
PATCHES_PER_SIZE = 20
N_REPEATS = 1

RANDOM_STATE = 42
MIN_CLEAN_NUCLEI_PER_PATCH = 50

# Individual geometry matchers only. No adaptive controller.
MATCHERS = ("quad", "triangles", "hashing")

# Missed nuclei / under-segmentation.
MISSED_NUCLEI_PCT = [0, 10, 20, 40, 60, 80, 90, 95]

# Extra detections / over-segmentation / spurious detections.
# 100% = add as many fake detections as true detections.
# 1000% = add 10x as many fake detections as true detections.
EXTRA_DETECTIONS_PCT = [0, 10, 25, 50, 100, 200, 500, 1000]

# Where to inject errors.
# "query" is easiest to interpret; "both" is more realistic but slower.
PERTURBATION_TARGETS = ("query", "both")
# PERTURBATION_TARGETS = ("query",)  # faster first pass

# Benchmark success threshold.
SSIM_SUCCESS_THRESH = 0.80

# Diagnostic only.
TRANSLATION_ERROR_WARN_PX = 20

# Cap reference false positives to avoid runaway memory/runtime.
# Query false positives are not capped because the query patch is small.
MAX_EXTRA_REFERENCE_POINTS = 100000

# Keep notebook output manageable.
QUIET_NUCLEISKY = True

# NucleiSky-valid permissive matcher overrides.
# NOTE: these are valid keys in the current repo config.
# Do not use min_frac_inliers_success / accept_transform_without_success here.
COMMON_MATCHER_OVERRIDES = {
    "use_dynamic_scale": False,
    "frac_inliers_thresh": 0.0,
    "use_icp_refinement": True,
}

MATCHER_SPECIFIC_OVERRIDES = {
    "quad": {
        "min_inliers_abs": 3,
        "min_inliers_frac": 0.0,
        "early_stop_frac": 1.0,
    },
    "triangles": {
        "min_inliers_abs": 3,
        "min_inliers_frac": 0.0,
        "early_stop_frac": 1.0,
    },
    "hashing": {
        "min_inliers_abs": 3,
        "min_inliers_frac": 0.0,
        "early_stop_frac": 1.0,
    },
}

MATCHER_CONFIG = None

# ---------------------------------------------------------------------
# 2. Retrieve notebook variables
# ---------------------------------------------------------------------

img_for_bench = globals().get("img_full", None)
if img_for_bench is None:
    img_for_bench = globals().get("img_full_seg", None)

if img_for_bench is None:
    raise RuntimeError("Could not find img_full or img_full_seg. Run the image-loading cells first.")

if "df_full" not in globals():
    raise RuntimeError("df_full not found. Run the segmentation/centroid extraction cells first.")

if "NucleiSky" not in globals():
    raise RuntimeError("NucleiSky not found. Run the NucleiSky setup/import cells first.")

if "ij_percentile_normalize" not in globals():
    raise RuntimeError("ij_percentile_normalize not found. Run the notebook setup cells first.")

pix_for_bench = float(
    globals().get(
        "PIXEL_SIZE_UM_FULL_SEG",
        globals().get("PIXEL_SIZE_UM", 1.0)
    )
)

img_for_bench = np.asarray(img_for_bench)
if img_for_bench.ndim != 2:
    raise ValueError(f"This benchmark expects a 2D image. Got shape={img_for_bench.shape}")

H, W = img_for_bench.shape

required_cols = ["centroid_y_px", "centroid_x_px", "centroid_y_um", "centroid_x_um"]
missing = [c for c in required_cols if c not in df_full.columns]
if missing:
    raise ValueError(f"df_full is missing required columns: {missing}")

if "RESULT_DIR" in globals() and RESULT_DIR:
    base_output_dir = Path(RESULT_DIR)
else:
    base_output_dir = Path("benchmark_results")

output_dir = base_output_dir / "segmentation_error_missed_extra_ssim_only_512px"
output_dir.mkdir(parents=True, exist_ok=True)

base_name = "current_image"
try:
    meta_path = base_output_dir / "inputs_metadata.json"
    if meta_path.exists():
        with open(meta_path, "r") as f:
            meta = json.load(f)
        if "full_image_path" in meta:
            base_name = Path(meta["full_image_path"]).stem
except Exception:
    pass

PARTIAL_CSV = output_dir / f"{base_name}_missed_extra_ssim_only_512_PARTIAL.csv"
RAW_CSV = output_dir / f"{base_name}_missed_extra_ssim_only_512_raw.csv"
SUMMARY_CSV = output_dir / f"{base_name}_missed_extra_ssim_only_512_summary.csv"
FAILURE_CSV = output_dir / f"{base_name}_missed_extra_ssim_only_512_failure_reasons.csv"
CONFIG_JSON = output_dir / f"{base_name}_missed_extra_ssim_only_512_config.json"

print("NucleiSky segmentation-error sensitivity benchmark")
print("Success criterion: SSIM only")
print("Adaptive mode: disabled")
print(f"Matchers: {MATCHERS}")
print(f"Image shape: {img_for_bench.shape}")
print(f"Pixel size: {pix_for_bench:.4f} um/px")
print(f"N nuclei in clean reference: {len(df_full)}")
print(f"Patch size: {PATCH_SIZE_PX}px")
print(f"Patch count: {PATCHES_PER_SIZE}")
print(f"Minimum clean nuclei per patch: {MIN_CLEAN_NUCLEI_PER_PATCH}")
print(f"Repeats: {N_REPEATS}")
print(f"Targets: {PERTURBATION_TARGETS}")
print(f"Saving to: {output_dir}")

# ---------------------------------------------------------------------
# 3. Resume/checkpoint setup
# ---------------------------------------------------------------------

current_config = {
    "patch_size_px": PATCH_SIZE_PX,
    "patches_per_size": PATCHES_PER_SIZE,
    "n_repeats": N_REPEATS,
    "random_state": RANDOM_STATE,
    "min_clean_nuclei_per_patch": MIN_CLEAN_NUCLEI_PER_PATCH,
    "matchers": list(MATCHERS),
    "missed_nuclei_pct": list(MISSED_NUCLEI_PCT),
    "extra_detections_pct": list(EXTRA_DETECTIONS_PCT),
    "perturbation_targets": list(PERTURBATION_TARGETS),
    "ssim_success_thresh": SSIM_SUCCESS_THRESH,
    "max_extra_reference_points": MAX_EXTRA_REFERENCE_POINTS,
    "pixel_size_um": pix_for_bench,
    "image_shape": list(img_for_bench.shape),
}

if CONFIG_JSON.exists():
    try:
        with open(CONFIG_JSON, "r") as f:
            previous_config = json.load(f)
        if previous_config != current_config:
            print("\nWARNING: Existing benchmark config differs from current settings.")
            print("The cell will still resume matching jobs by key, but this may mix runs if settings changed.")
            print("Use a new output folder or delete previous CSVs if you want a clean run.\n")
    except Exception as e:
        print(f"Could not read previous config file: {e}")

with open(CONFIG_JSON, "w") as f:
    json.dump(current_config, f, indent=2)

def _safe_float(v):
    try:
        return float(v)
    except Exception:
        return np.nan

def _safe_int(v):
    try:
        return int(v)
    except Exception:
        return -1

def _normalise_matcher_name(m):
    m = str(m).lower()
    if m == "triangle":
        return "triangles"
    return m

def _job_key_from_values(error_mode, error_pct, target, repeat, patch_idx, matcher):
    return (
        str(error_mode),
        float(error_pct),
        str(target),
        int(repeat),
        int(patch_idx),
        _normalise_matcher_name(matcher),
    )

def _job_key_from_row(row):
    return _job_key_from_values(
        row.get("error_mode"),
        row.get("error_pct"),
        row.get("perturbation_target"),
        row.get("repeat"),
        row.get("patch_idx"),
        row.get("matcher"),
    )

resume_source = None
if PARTIAL_CSV.exists():
    resume_source = PARTIAL_CSV
elif RAW_CSV.exists():
    resume_source = RAW_CSV

if resume_source is not None:
    df_existing = pd.read_csv(resume_source)
    rows = df_existing.to_dict("records")

    completed_jobs = set()
    for r in rows:
        try:
            completed_jobs.add(_job_key_from_row(r))
        except Exception:
            pass

    print(f"\nResuming from existing benchmark data:")
    print(resume_source)
    print(f"Reloaded rows: {len(rows)}")
    print(f"Completed matcher jobs: {len(completed_jobs)}\n")
else:
    rows = []
    completed_jobs = set()
    print("\nNo previous benchmark data found in this output folder.")
    print("Starting from scratch.\n")

# ---------------------------------------------------------------------
# 4. Helper functions
# ---------------------------------------------------------------------

def _stable_u32_from_values(*values):
    s = "|".join(map(str, values))
    return zlib.adler32(s.encode("utf-8")) & 0xFFFFFFFF


def _minimal_centroid_df(df):
    """
    Keep only columns needed for this benchmark.
    Dynamic scale is disabled, so we do not need area/bbox/feature columns.
    """
    out = df[["centroid_y_px", "centroid_x_px", "centroid_y_um", "centroid_x_um"]].copy()
    out["is_synthetic_fp"] = False
    return out.reset_index(drop=True)


def perturb_centroids_missed_or_extra(
    df,
    *,
    pixel_size_um,
    image_shape_px,
    missed_fraction=0.0,
    extra_fraction=0.0,
    rng=None,
    max_extra_points=None,
):
    """
    Simulate one dominant segmentation error mode at a time.

    missed_fraction:
        Fraction of existing nuclei to remove.
        Example: 0.80 removes 80% of detections.

    extra_fraction:
        Extra false detections as a fraction of the current detections.
        Example: 1.00 adds 1x extra detections.
        Example: 10.00 adds 10x extra detections.

    Fake detections are sampled uniformly within the relevant image domain:
        - query patch domain for query perturbation
        - full reference domain for reference perturbation
    """
    if rng is None:
        rng = np.random.default_rng(0)

    df = _minimal_centroid_df(df)
    height_px, width_px = int(image_shape_px[0]), int(image_shape_px[1])

    if len(df) == 0:
        df.attrs["n_extra_requested"] = 0
        df.attrs["n_extra_added"] = 0
        df.attrs["extra_capped"] = False
        return df

    # Missed nuclei / under-segmentation.
    if missed_fraction > 0:
        keep = rng.random(len(df)) >= float(missed_fraction)
        df = df.loc[keep].copy().reset_index(drop=True)

    # Extra detections / over-segmentation / spurious detections.
    n_real_after_missed = len(df)
    n_extra_requested = int(round(float(extra_fraction) * max(1, n_real_after_missed)))
    n_extra = n_extra_requested

    if max_extra_points is not None:
        n_extra = min(n_extra, int(max_extra_points))

    if n_extra > 0:
        y_extra = rng.uniform(0, max(1, height_px), size=n_extra)
        x_extra = rng.uniform(0, max(1, width_px), size=n_extra)

        y_extra = np.clip(y_extra, 0, max(0, height_px - 1))
        x_extra = np.clip(x_extra, 0, max(0, width_px - 1))

        fp_df = pd.DataFrame({
            "centroid_y_px": y_extra,
            "centroid_x_px": x_extra,
            "centroid_y_um": y_extra * float(pixel_size_um),
            "centroid_x_um": x_extra * float(pixel_size_um),
            "is_synthetic_fp": True,
        })

        df = pd.concat([df, fp_df], ignore_index=True)

    out = df.reset_index(drop=True)
    out.attrs["n_extra_requested"] = n_extra_requested
    out.attrs["n_extra_added"] = n_extra
    out.attrs["extra_capped"] = bool(n_extra < n_extra_requested)

    return out


def make_512_patch_plan():
    if "make_benchmark_patch_plan" in globals():
        plan = make_benchmark_patch_plan(
            img_full=img_for_bench,
            patch_sizes_px=[PATCH_SIZE_PX],
            patches_per_size=PATCHES_PER_SIZE,
            seed=RANDOM_STATE,
            df_full=df_full,
            min_nuclei=MIN_CLEAN_NUCLEI_PER_PATCH,
        )
        return list(plan[PATCH_SIZE_PX])

    # Fallback if make_benchmark_patch_plan is unavailable.
    rng = np.random.default_rng(RANDOM_STATE)
    candidates = []
    nuclei = df_full[["centroid_y_px", "centroid_x_px"]].to_numpy(dtype=float)

    attempts = 0
    max_attempts = PATCHES_PER_SIZE * 1000
    while len(candidates) < PATCHES_PER_SIZE and attempts < max_attempts:
        attempts += 1

        y_c, x_c = nuclei[rng.integers(0, len(nuclei))]
        y0 = int(np.clip(round(y_c - rng.uniform(0, PATCH_SIZE_PX)), 0, H - PATCH_SIZE_PX))
        x0 = int(np.clip(round(x_c - rng.uniform(0, PATCH_SIZE_PX)), 0, W - PATCH_SIZE_PX))

        if (y0, x0) in candidates:
            continue

        n_patch = (
            (df_full["centroid_y_px"] >= y0) &
            (df_full["centroid_y_px"] < y0 + PATCH_SIZE_PX) &
            (df_full["centroid_x_px"] >= x0) &
            (df_full["centroid_x_px"] < x0 + PATCH_SIZE_PX)
        ).sum()

        if n_patch >= MIN_CLEAN_NUCLEI_PER_PATCH:
            candidates.append((y0, x0))

    return candidates


def make_valid_matcher_kwargs(matcher, seed):
    """
    Build matcher_kwargs using ONLY keys that exist in the current NucleiSky config.
    """
    matcher = str(matcher).lower()
    if matcher == "triangle":
        matcher = "triangles"

    return {
        "_common": {
            **COMMON_MATCHER_OVERRIDES,
            "random_state": int(seed),
        },
        matcher: {
            **MATCHER_SPECIFIC_OVERRIDES.get(matcher, {}),
        },
    }


def run_matcher_for_segerr(
    *,
    matcher,
    centroids_crop_um,
    centroids_full_um,
    img_crop,
    random_seed,
):
    """
    Run an individual NucleiSky matcher and return raw NucleiSky output.

    For this benchmark, out["success"] is NOT used as benchmark success.
    We only need best_t, then score the predicted crop using SSIM.
    """
    matcher = str(matcher).lower()

    matcher_kwargs = make_valid_matcher_kwargs(matcher, random_seed)

    call = lambda: NucleiSky(
        centroids_crop_um=centroids_crop_um,
        centroids_full_um=centroids_full_um,
        img_full=img_for_bench,
        img_crop=img_crop,
        pixel_size_full_um=float(pix_for_bench),
        pixel_size_crop_um=float(pix_for_bench),
        matcher=matcher,
        ij_percentile_normalize=ij_percentile_normalize,
        features_crop=None,
        features_full=None,
        df_full=None,
        df_crop=None,
        matcher_config=MATCHER_CONFIG,
        matcher_kwargs=matcher_kwargs,
        save_dir=None,
        save_prefix=f"segerr_ssim_only_{matcher}",
    )

    if QUIET_NUCLEISKY:
        with contextlib.redirect_stdout(io.StringIO()):
            return call()

    return call()


def extract_match_quality(out):
    q = {}
    if isinstance(out, dict):
        q = out.get("match_quality", {}) or {}

    try:
        frac = float(q.get("frac_inliers", np.nan))
    except Exception:
        frac = np.nan

    try:
        mean_err = q.get("mean_error_um", np.nan)
        mean_err = float(mean_err) if mean_err is not None else np.nan
    except Exception:
        mean_err = np.nan

    return frac, mean_err


def compute_ssim_and_translation(out, patch_img, patch_size_px, y0, x0):
    """
    Use best_t even if NucleiSky reports success=False.
    Benchmark success is computed from SSIM only.
    """
    if out is None or not isinstance(out, dict) or out.get("best_t", None) is None:
        return False, np.nan, np.nan, np.nan, np.nan

    best_t = np.asarray(out["best_t"], dtype=float).reshape(2,)

    true_topleft_um = np.array(
        [y0 * float(pix_for_bench), x0 * float(pix_for_bench)],
        dtype=float,
    )

    trans_error_px = float(np.linalg.norm(best_t - true_topleft_um) / float(pix_for_bench))

    y_pred0 = int(round(best_t[0] / float(pix_for_bench)))
    x_pred0 = int(round(best_t[1] / float(pix_for_bench)))

    if not ((0 <= y_pred0 <= H - patch_size_px) and (0 <= x_pred0 <= W - patch_size_px)):
        return True, np.nan, trans_error_px, y_pred0, x_pred0

    pred_roi = img_for_bench[
        y_pred0:y_pred0 + patch_size_px,
        x_pred0:x_pred0 + patch_size_px,
    ]

    patch_norm = ij_percentile_normalize(patch_img)
    pred_norm = ij_percentile_normalize(pred_roi)

    ssim_val = float(ssim(patch_norm, pred_norm, data_range=1.0))

    return True, ssim_val, trans_error_px, y_pred0, x_pred0


def append_row_and_mark(row):
    rows.append(row)
    try:
        completed_jobs.add(_job_key_from_row(row))
    except Exception:
        pass


def save_partial():
    partial_df = pd.DataFrame(rows)
    partial_df.to_csv(PARTIAL_CSV, index=False)
    return partial_df


# ---------------------------------------------------------------------
# 5. Define benchmark conditions
# ---------------------------------------------------------------------

conditions = []

for pct in MISSED_NUCLEI_PCT:
    conditions.append({
        "error_mode": "missed_nuclei",
        "error_pct": float(pct),
        "missed_fraction": float(pct) / 100.0,
        "extra_fraction": 0.0,
    })

for pct in EXTRA_DETECTIONS_PCT:
    conditions.append({
        "error_mode": "extra_detections",
        "error_pct": float(pct),
        "missed_fraction": 0.0,
        "extra_fraction": float(pct) / 100.0,
    })

patch_candidates = make_512_patch_plan()
print(f"Using {len(patch_candidates)} query patches of size {PATCH_SIZE_PX}px.")

if len(patch_candidates) == 0:
    raise RuntimeError("No suitable patches found. Try lowering MIN_CLEAN_NUCLEI_PER_PATCH or PATCHES_PER_SIZE.")

# ---------------------------------------------------------------------
# 6. Main benchmark loop
# ---------------------------------------------------------------------

n_skipped = 0
n_new = 0

for cond in conditions:
    error_mode = cond["error_mode"]
    error_pct = cond["error_pct"]

    for target in PERTURBATION_TARGETS:
        for repeat in range(N_REPEATS):

            ref_seed = _stable_u32_from_values(
                RANDOM_STATE, "reference", error_mode, error_pct, target, repeat
            )
            ref_rng = np.random.default_rng(ref_seed)

            # Perturb full reference only when target includes reference.
            if target in ("reference", "both"):
                df_ref = perturb_centroids_missed_or_extra(
                    df_full,
                    pixel_size_um=pix_for_bench,
                    image_shape_px=img_for_bench.shape,
                    missed_fraction=cond["missed_fraction"],
                    extra_fraction=cond["extra_fraction"],
                    rng=ref_rng,
                    max_extra_points=MAX_EXTRA_REFERENCE_POINTS if error_mode == "extra_detections" else None,
                )
                ref_extra_requested = int(df_ref.attrs.get("n_extra_requested", 0))
                ref_extra_added = int(df_ref.attrs.get("n_extra_added", 0))
                ref_extra_capped = bool(df_ref.attrs.get("extra_capped", False))
            else:
                df_ref = _minimal_centroid_df(df_full)
                ref_extra_requested = 0
                ref_extra_added = 0
                ref_extra_capped = False

            centroids_full_um = df_ref[["centroid_y_um", "centroid_x_um"]].to_numpy(dtype=float, copy=False)

            for patch_idx, (y0, x0) in enumerate(
                tqdm(
                    patch_candidates,
                    desc=f"{error_mode} {error_pct:g}% | target={target} | repeat {repeat + 1}/{N_REPEATS}",
                    leave=False,
                )
            ):
                y0 = int(y0)
                x0 = int(x0)
                size = int(PATCH_SIZE_PX)

                patch_img = img_for_bench[y0:y0 + size, x0:x0 + size]

                clean_mask = (
                    (df_full["centroid_y_px"] >= y0) &
                    (df_full["centroid_y_px"] < y0 + size) &
                    (df_full["centroid_x_px"] >= x0) &
                    (df_full["centroid_x_px"] < x0 + size)
                )

                df_query_clean = _minimal_centroid_df(df_full.loc[clean_mask])
                n_query_clean = int(len(df_query_clean))

                # Convert query centroids to patch-local coordinates.
                df_query_clean["centroid_y_px"] = df_query_clean["centroid_y_px"].astype(float) - float(y0)
                df_query_clean["centroid_x_px"] = df_query_clean["centroid_x_px"].astype(float) - float(x0)
                df_query_clean["centroid_y_um"] = df_query_clean["centroid_y_px"] * float(pix_for_bench)
                df_query_clean["centroid_x_um"] = df_query_clean["centroid_x_px"] * float(pix_for_bench)

                query_seed = _stable_u32_from_values(
                    RANDOM_STATE, "query", error_mode, error_pct, target, repeat, patch_idx, y0, x0
                )
                query_rng = np.random.default_rng(query_seed)

                # Perturb query only when target includes query.
                if target in ("query", "both"):
                    df_query = perturb_centroids_missed_or_extra(
                        df_query_clean,
                        pixel_size_um=pix_for_bench,
                        image_shape_px=(size, size),
                        missed_fraction=cond["missed_fraction"],
                        extra_fraction=cond["extra_fraction"],
                        rng=query_rng,
                        max_extra_points=None,
                    )
                    query_extra_requested = int(df_query.attrs.get("n_extra_requested", 0))
                    query_extra_added = int(df_query.attrs.get("n_extra_added", 0))
                    query_extra_capped = bool(df_query.attrs.get("extra_capped", False))
                else:
                    df_query = df_query_clean.copy().reset_index(drop=True)
                    query_extra_requested = 0
                    query_extra_added = 0
                    query_extra_capped = False

                n_query_after = int(len(df_query))
                n_ref_after = int(len(df_ref))

                n_query_fake = int(df_query["is_synthetic_fp"].fillna(False).sum())
                n_ref_fake = int(df_ref["is_synthetic_fp"].fillna(False).sum())

                actual_query_extra_pct = 100.0 * n_query_fake / max(1, n_query_after - n_query_fake)
                actual_ref_extra_pct = 100.0 * n_ref_fake / max(1, n_ref_after - n_ref_fake)

                centroids_crop_um = None
                if n_query_after >= 3 and n_ref_after >= 3:
                    centroids_crop_um = df_query[["centroid_y_um", "centroid_x_um"]].to_numpy(dtype=float, copy=False)

                for matcher in MATCHERS:
                    matcher = _normalise_matcher_name(matcher)

                    job_key = _job_key_from_values(
                        error_mode, error_pct, target, repeat, patch_idx, matcher
                    )

                    if job_key in completed_jobs:
                        n_skipped += 1
                        continue

                    # At least 3 points are required to estimate a 2D similarity transform.
                    if n_query_after < 3 or n_ref_after < 3:
                        row = {
                            "error_mode": error_mode,
                            "error_pct": error_pct,
                            "perturbation_target": target,
                            "repeat": repeat,
                            "patch_idx": patch_idx,
                            "patch_y0": y0,
                            "patch_x0": x0,
                            "patch_size_px": size,
                            "matcher": matcher,
                            "n_query_clean": n_query_clean,
                            "n_query_after": n_query_after,
                            "n_reference_after": n_ref_after,
                            "n_query_fake": n_query_fake,
                            "n_reference_fake": n_ref_fake,
                            "query_extra_requested": query_extra_requested,
                            "query_extra_added": query_extra_added,
                            "query_extra_capped": query_extra_capped,
                            "reference_extra_requested": ref_extra_requested,
                            "reference_extra_added": ref_extra_added,
                            "reference_extra_capped": ref_extra_capped,
                            "actual_query_extra_pct": actual_query_extra_pct,
                            "actual_reference_extra_pct": actual_ref_extra_pct,
                            "has_transform": False,
                            "matcher_reported_success": False,
                            "frac_inliers_diagnostic": np.nan,
                            "mean_error_um_diagnostic": np.nan,
                            "ssim": np.nan,
                            "ssim_success": False,
                            "trans_error_px_diagnostic": np.nan,
                            "translation_error_warn": False,
                            "y_pred0": np.nan,
                            "x_pred0": np.nan,
                            "time_s": 0.0,
                            "failure_reason": "too_few_nuclei_after_perturbation",
                        }
                        append_row_and_mark(row)
                        n_new += 1
                        continue

                    run_seed = _stable_u32_from_values(
                        RANDOM_STATE, "run", error_mode, error_pct, target, repeat, patch_idx, matcher
                    )

                    t0 = time.perf_counter()
                    failure_reason = ""

                    try:
                        out = run_matcher_for_segerr(
                            matcher=matcher,
                            centroids_crop_um=centroids_crop_um,
                            centroids_full_um=centroids_full_um,
                            img_crop=patch_img,
                            random_seed=run_seed,
                        )

                        elapsed = float(time.perf_counter() - t0)

                        has_transform, ssim_val, trans_error_px, y_pred0, x_pred0 = compute_ssim_and_translation(
                            out,
                            patch_img,
                            size,
                            y0,
                            x0,
                        )

                        matcher_reported_success = bool(out.get("success", False)) if isinstance(out, dict) else False
                        frac_diag, mean_err_diag = extract_match_quality(out)

                        # The only benchmark success criterion.
                        ssim_success = bool(np.isfinite(ssim_val) and ssim_val >= SSIM_SUCCESS_THRESH)

                        if not has_transform:
                            failure_reason = "no_transform"
                        elif not np.isfinite(ssim_val):
                            failure_reason = "transform_out_of_bounds_or_invalid_ssim"
                        elif not ssim_success:
                            failure_reason = "ssim_below_threshold"

                        row = {
                            "error_mode": error_mode,
                            "error_pct": error_pct,
                            "perturbation_target": target,
                            "repeat": repeat,
                            "patch_idx": patch_idx,
                            "patch_y0": y0,
                            "patch_x0": x0,
                            "patch_size_px": size,
                            "matcher": matcher,
                            "n_query_clean": n_query_clean,
                            "n_query_after": n_query_after,
                            "n_reference_after": n_ref_after,
                            "n_query_fake": n_query_fake,
                            "n_reference_fake": n_ref_fake,
                            "query_extra_requested": query_extra_requested,
                            "query_extra_added": query_extra_added,
                            "query_extra_capped": query_extra_capped,
                            "reference_extra_requested": ref_extra_requested,
                            "reference_extra_added": ref_extra_added,
                            "reference_extra_capped": ref_extra_capped,
                            "actual_query_extra_pct": actual_query_extra_pct,
                            "actual_reference_extra_pct": actual_ref_extra_pct,
                            "has_transform": bool(has_transform),
                            "matcher_reported_success": matcher_reported_success,
                            "frac_inliers_diagnostic": frac_diag,
                            "mean_error_um_diagnostic": mean_err_diag,
                            "ssim": ssim_val,
                            "ssim_success": ssim_success,
                            "trans_error_px_diagnostic": trans_error_px,
                            "translation_error_warn": bool(
                                np.isfinite(trans_error_px) and trans_error_px >= TRANSLATION_ERROR_WARN_PX
                            ),
                            "y_pred0": y_pred0,
                            "x_pred0": x_pred0,
                            "time_s": elapsed,
                            "failure_reason": failure_reason,
                        }

                        append_row_and_mark(row)
                        n_new += 1

                    except Exception as e:
                        elapsed = float(time.perf_counter() - t0)
                        row = {
                            "error_mode": error_mode,
                            "error_pct": error_pct,
                            "perturbation_target": target,
                            "repeat": repeat,
                            "patch_idx": patch_idx,
                            "patch_y0": y0,
                            "patch_x0": x0,
                            "patch_size_px": size,
                            "matcher": matcher,
                            "n_query_clean": n_query_clean,
                            "n_query_after": n_query_after,
                            "n_reference_after": n_ref_after,
                            "n_query_fake": n_query_fake,
                            "n_reference_fake": n_ref_fake,
                            "query_extra_requested": query_extra_requested,
                            "query_extra_added": query_extra_added,
                            "query_extra_capped": query_extra_capped,
                            "reference_extra_requested": ref_extra_requested,
                            "reference_extra_added": ref_extra_added,
                            "reference_extra_capped": ref_extra_capped,
                            "actual_query_extra_pct": actual_query_extra_pct,
                            "actual_reference_extra_pct": actual_ref_extra_pct,
                            "has_transform": False,
                            "matcher_reported_success": False,
                            "frac_inliers_diagnostic": np.nan,
                            "mean_error_um_diagnostic": np.nan,
                            "ssim": np.nan,
                            "ssim_success": False,
                            "trans_error_px_diagnostic": np.nan,
                            "translation_error_warn": False,
                            "y_pred0": np.nan,
                            "x_pred0": np.nan,
                            "time_s": elapsed,
                            "failure_reason": f"exception: {type(e).__name__}: {e}",
                        }

                        append_row_and_mark(row)
                        n_new += 1

            save_partial()

    save_partial()
    print(f"Checkpoint saved after {error_mode} {error_pct:g}%. New jobs: {n_new}; skipped jobs: {n_skipped}")

# ---------------------------------------------------------------------
# 7. Save raw, summary and failure outputs
# ---------------------------------------------------------------------

df_segerr = pd.DataFrame(rows)

# Remove accidental exact duplicate rows if any prior interrupted run duplicated a job.
# Keep the last row for each matcher job.
dedupe_cols = [
    "error_mode",
    "error_pct",
    "perturbation_target",
    "repeat",
    "patch_idx",
    "matcher",
]

if len(df_segerr) > 0:
    df_segerr["matcher"] = df_segerr["matcher"].map(_normalise_matcher_name)
    df_segerr = df_segerr.drop_duplicates(subset=dedupe_cols, keep="last").reset_index(drop=True)

df_segerr.to_csv(RAW_CSV, index=False)
df_segerr.to_csv(PARTIAL_CSV, index=False)

summary_cols = [
    "error_mode",
    "error_pct",
    "perturbation_target",
    "matcher",
]

def _q25(x):
    x = np.asarray(x, float)
    return np.nanpercentile(x, 25) if np.isfinite(x).any() else np.nan

def _q75(x):
    x = np.asarray(x, float)
    return np.nanpercentile(x, 75) if np.isfinite(x).any() else np.nan

if len(df_segerr) > 0:
    df_summary = (
        df_segerr
        .groupby(summary_cols, dropna=False)
        .agg(
            n_trials=("ssim_success", "size"),

            # Main endpoint for this benchmark:
            ssim_success_rate=("ssim_success", "mean"),

            # Continuous image-level validation:
            median_ssim=("ssim", "median"),
            q25_ssim=("ssim", _q25),
            q75_ssim=("ssim", _q75),

            # Diagnostics only:
            median_trans_error_px=("trans_error_px_diagnostic", "median"),
            q25_trans_error_px=("trans_error_px_diagnostic", _q25),
            q75_trans_error_px=("trans_error_px_diagnostic", _q75),
            median_frac_inliers_diagnostic=("frac_inliers_diagnostic", "median"),
            median_mean_error_um_diagnostic=("mean_error_um_diagnostic", "median"),

            median_query_nuclei_clean=("n_query_clean", "median"),
            median_query_nuclei_after=("n_query_after", "median"),
            median_reference_nuclei_after=("n_reference_after", "median"),
            median_query_fake=("n_query_fake", "median"),
            median_reference_fake=("n_reference_fake", "median"),
            median_actual_query_extra_pct=("actual_query_extra_pct", "median"),
            median_actual_reference_extra_pct=("actual_reference_extra_pct", "median"),

            transform_rate=("has_transform", "mean"),
            matcher_reported_success_rate=("matcher_reported_success", "mean"),
            median_time_s=("time_s", "median"),
            reference_extra_capped_any=("reference_extra_capped", "max"),
        )
        .reset_index()
    )
else:
    df_summary = pd.DataFrame()

df_summary.to_csv(SUMMARY_CSV, index=False)

if len(df_segerr) > 0:
    failure_summary = (
        df_segerr
        .groupby(
            ["error_mode", "error_pct", "perturbation_target", "matcher", "failure_reason"],
            dropna=False,
        )
        .size()
        .reset_index(name="n")
        .sort_values(["error_mode", "perturbation_target", "matcher", "error_pct", "n"])
    )
else:
    failure_summary = pd.DataFrame()

failure_summary.to_csv(FAILURE_CSV, index=False)

print(f"Saved raw results:\n{RAW_CSV}")
print(f"Saved partial checkpoint:\n{PARTIAL_CSV}")
print(f"Saved summary:\n{SUMMARY_CSV}")
print(f"Saved failure summary:\n{FAILURE_CSV}")

display(df_summary.head(40))
display(failure_summary.head(40))

print("\nBenchmark complete.")
print(f"New matcher jobs computed in this run: {n_new}")
print(f"Previously completed matcher jobs skipped: {n_skipped}")
print(f"Total unique matcher jobs in table: {len(df_segerr)}")

In [ ]:
#@title Reload segmentation-error benchmark results

from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

def load_segmentation_benchmark_results(result_folder, base_name=None):
    """
    Reload segmentation-error benchmark outputs from a result folder.

    result_folder should point to:
        .../segmentation_error_missed_extra_ssim_only_512px

    If base_name is None, the function tries to infer it from *_raw.csv.
    """
    result_folder = Path(result_folder).expanduser().resolve()

    if not result_folder.exists():
        raise FileNotFoundError(f"Result folder does not exist: {result_folder}")

    raw_files = sorted(result_folder.glob("*_missed_extra_ssim_only_512_raw.csv"))
    partial_files = sorted(result_folder.glob("*_missed_extra_ssim_only_512_PARTIAL.csv"))
    summary_files = sorted(result_folder.glob("*_missed_extra_ssim_only_512_summary.csv"))
    failure_files = sorted(result_folder.glob("*_missed_extra_ssim_only_512_failure_reasons.csv"))

    if base_name is None:
        if raw_files:
            raw_path = raw_files[0]
            base_name = raw_path.name.replace("_missed_extra_ssim_only_512_raw.csv", "")
        elif partial_files:
            raw_path = partial_files[0]
            base_name = raw_path.name.replace("_missed_extra_ssim_only_512_PARTIAL.csv", "")
        else:
            raise FileNotFoundError(
                "Could not find either *_missed_extra_ssim_only_512_raw.csv "
                "or *_missed_extra_ssim_only_512_PARTIAL.csv in the folder."
            )
    else:
        raw_path = result_folder / f"{base_name}_missed_extra_ssim_only_512_raw.csv"

    partial_path = result_folder / f"{base_name}_missed_extra_ssim_only_512_PARTIAL.csv"
    summary_path = result_folder / f"{base_name}_missed_extra_ssim_only_512_summary.csv"
    failure_path = result_folder / f"{base_name}_missed_extra_ssim_only_512_failure_reasons.csv"

    # Prefer raw final file, fall back to partial checkpoint.
    if raw_path.exists():
        df_segerr_loaded = pd.read_csv(raw_path)
        loaded_from = raw_path
    elif partial_path.exists():
        df_segerr_loaded = pd.read_csv(partial_path)
        loaded_from = partial_path
    else:
        raise FileNotFoundError(
            f"No raw or partial segmentation benchmark file found for base_name={base_name}"
        )

    if summary_path.exists():
        df_summary_loaded = pd.read_csv(summary_path)
    else:
        df_summary_loaded = summarise_segmentation_benchmark(df_segerr_loaded)
        df_summary_loaded.to_csv(summary_path, index=False)

    if failure_path.exists():
        failure_summary_loaded = pd.read_csv(failure_path)
    else:
        failure_summary_loaded = make_segmentation_failure_summary(df_segerr_loaded)
        failure_summary_loaded.to_csv(failure_path, index=False)

    print(f"Loaded segmentation benchmark from:\n{loaded_from}")
    print(f"Rows: {len(df_segerr_loaded)}")
    print(f"Summary rows: {len(df_summary_loaded)}")
    print(f"Failure summary rows: {len(failure_summary_loaded)}")

    return df_segerr_loaded, df_summary_loaded, failure_summary_loaded, result_folder, base_name


def summarise_segmentation_benchmark(df_segerr):
    df = df_segerr.copy()

    for col in ["ssim_success", "has_transform", "matcher_reported_success", "reference_extra_capped"]:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)

    numeric_cols = [
        "error_pct", "ssim", "trans_error_px_diagnostic",
        "frac_inliers_diagnostic", "mean_error_um_diagnostic",
        "n_query_clean", "n_query_after", "n_reference_after",
        "n_query_fake", "n_reference_fake",
        "actual_query_extra_pct", "actual_reference_extra_pct",
        "time_s",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    def _q25(x):
        x = pd.to_numeric(x, errors="coerce").to_numpy(dtype=float)
        return np.nanpercentile(x, 25) if np.isfinite(x).any() else np.nan

    def _q75(x):
        x = pd.to_numeric(x, errors="coerce").to_numpy(dtype=float)
        return np.nanpercentile(x, 75) if np.isfinite(x).any() else np.nan

    summary_cols = ["error_mode", "error_pct", "perturbation_target", "matcher"]

    return (
        df
        .groupby(summary_cols, dropna=False)
        .agg(
            n_trials=("ssim_success", "size"),
            ssim_success_rate=("ssim_success", "mean"),
            median_ssim=("ssim", "median"),
            q25_ssim=("ssim", _q25),
            q75_ssim=("ssim", _q75),
            median_trans_error_px=("trans_error_px_diagnostic", "median"),
            q25_trans_error_px=("trans_error_px_diagnostic", _q25),
            q75_trans_error_px=("trans_error_px_diagnostic", _q75),
            median_frac_inliers_diagnostic=("frac_inliers_diagnostic", "median"),
            median_mean_error_um_diagnostic=("mean_error_um_diagnostic", "median"),
            median_query_nuclei_clean=("n_query_clean", "median"),
            median_query_nuclei_after=("n_query_after", "median"),
            median_reference_nuclei_after=("n_reference_after", "median"),
            median_query_fake=("n_query_fake", "median"),
            median_reference_fake=("n_reference_fake", "median"),
            median_actual_query_extra_pct=("actual_query_extra_pct", "median"),
            median_actual_reference_extra_pct=("actual_reference_extra_pct", "median"),
            transform_rate=("has_transform", "mean"),
            matcher_reported_success_rate=("matcher_reported_success", "mean"),
            median_time_s=("time_s", "median"),
            reference_extra_capped_any=("reference_extra_capped", "max"),
        )
        .reset_index()
    )


def make_segmentation_failure_summary(df_segerr):
    df = df_segerr.copy()

    if "failure_reason" not in df.columns:
        df["failure_reason"] = ""

    return (
        df
        .groupby(
            ["error_mode", "error_pct", "perturbation_target", "matcher", "failure_reason"],
            dropna=False,
        )
        .size()
        .reset_index(name="n")
        .sort_values(["error_mode", "perturbation_target", "matcher", "error_pct", "n"])
    )


# Try to pre-fill with the expected folder if RESULT_DIR exists.
default_folder = ""
if "RESULT_DIR" in globals() and RESULT_DIR:
    default_folder = str(Path(RESULT_DIR) / "segmentation_error_missed_extra_ssim_only_512px")

folder_input = widgets.Text(
    value=default_folder,
    placeholder="Path to segmentation_error_missed_extra_ssim_only_512px",
    description="Folder:",
    layout=widgets.Layout(width="75%")
)

base_input = widgets.Text(
    value="",
    placeholder="Optional base_name; leave empty to auto-detect",
    description="Base:",
    layout=widgets.Layout(width="75%")
)

load_button = widgets.Button(
    description="Load results",
    button_style="info",
    tooltip="Reload segmentation benchmark results"
)

output_area = widgets.Output()

def on_load_clicked(b):
    with output_area:
        output_area.clear_output()

        folder = folder_input.value.strip()
        base = base_input.value.strip() or None

        if not folder:
            print("Please enter a result folder.")
            return

        try:
            global df_segerr, df_summary, failure_summary, SEGERR_OUTPUT_DIR, SEGERR_BASE_NAME

            df_segerr, df_summary, failure_summary, SEGERR_OUTPUT_DIR, SEGERR_BASE_NAME = (
                load_segmentation_benchmark_results(folder, base_name=base)
            )

            display(df_summary.head(30))
            display(failure_summary.head(30))

        except Exception as e:
            print(f"Error loading segmentation benchmark results:\n{type(e).__name__}: {e}")

load_button.on_click(on_load_clicked)

display(widgets.VBox([
    folder_input,
    base_input,
    load_button,
    output_area,
]))

In [ ]:
#@title Plot segmentation-error robustness in a biology-friendly way

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from IPython.display import display, Markdown

# ---------------------------------------------------------------------
# 1. Plot settings
# ---------------------------------------------------------------------

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"

# Success definition for this benchmark:
# A trial succeeds if the predicted crop matches the original query crop by SSIM.
SSIM_SUCCESS_THRESH = 0.80
TARGET_PROB = 0.90

# Main figure: query-only perturbation is easiest to interpret.
MAIN_TARGET = "query"

# Also plot the harsher condition where both query and reference are perturbed.
PLOT_BOTH_TARGET = True

# Output directory
if "SEGERR_OUTPUT_DIR" in globals():
    plot_output_dir = Path(SEGERR_OUTPUT_DIR) / "bio_friendly_probability_plots"
elif "output_dir" in globals():
    plot_output_dir = Path(output_dir) / "bio_friendly_probability_plots"
else:
    plot_output_dir = Path("segmentation_error_bio_friendly_probability_plots")

plot_output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# 1b. Consistent matcher colours
# ---------------------------------------------------------------------

PREFERRED_MATCHER_ORDER = [
    "hashing",
    "graph",
    "quad",
    "triangles",
    "adaptive",
]

FIXED_MATCHER_COLORS = {
    "hashing": "#009E73",    # green
    "graph": "#E69F00",      # orange
    "quad": "#D62728",       # red
    "triangles": "#CC79A7",  # purple
    "adaptive": "#0072B2",   # blue
}

def normalise_matcher_name(matcher):
    matcher = str(matcher).strip().lower()

    aliases = {
        "triangle": "triangles",
        "tri": "triangles",
        "geo_hashing": "hashing",
        "geometric_hashing": "hashing",
        "geometric hashing": "hashing",
        "quad_matcher": "quad",
        "triangle_matcher": "triangles",
        "graph_matcher": "graph",
        "hashing_matcher": "hashing",
    }

    return aliases.get(matcher, matcher)


def get_matcher_order(df):
    present = sorted(df["matcher"].dropna().map(normalise_matcher_name).unique())
    ordered = [m for m in PREFERRED_MATCHER_ORDER if m in present]
    ordered += [m for m in present if m not in ordered]
    return ordered


def make_matcher_color_map(matchers):
    color_map = {}

    for matcher in matchers:
        if matcher in FIXED_MATCHER_COLORS:
            color_map[matcher] = FIXED_MATCHER_COLORS[matcher]

    unknown = [m for m in matchers if m not in color_map]
    if unknown:
        fallback = plt.get_cmap("tab10")
        for i, matcher in enumerate(unknown):
            color_map[matcher] = fallback(i % 10)

    return color_map


# ---------------------------------------------------------------------
# 2. Validate and prepare data
# ---------------------------------------------------------------------

if "df_segerr" not in globals():
    raise RuntimeError(
        "df_segerr not found. Run the benchmark cell or the reload-results cell first."
    )

df = df_segerr.copy()

required_cols = [
    "error_mode",
    "error_pct",
    "perturbation_target",
    "matcher",
    "ssim_success",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"df_segerr is missing required columns: {missing}")

def _as_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return bool(x)
    return str(x).strip().lower() in ["true", "1", "yes", "y"]

df["matcher"] = df["matcher"].map(normalise_matcher_name)
df["ssim_success"] = df["ssim_success"].map(_as_bool)
df["error_pct"] = pd.to_numeric(df["error_pct"], errors="coerce")

if "ssim" in df.columns:
    df["ssim"] = pd.to_numeric(df["ssim"], errors="coerce")
else:
    df["ssim"] = np.nan

if "has_transform" in df.columns:
    df["has_transform"] = df["has_transform"].map(_as_bool)
else:
    df["has_transform"] = False

if "actual_query_extra_pct" in df.columns:
    df["actual_query_extra_pct"] = pd.to_numeric(df["actual_query_extra_pct"], errors="coerce")
else:
    df["actual_query_extra_pct"] = np.nan

if "actual_reference_extra_pct" in df.columns:
    df["actual_reference_extra_pct"] = pd.to_numeric(df["actual_reference_extra_pct"], errors="coerce")
else:
    df["actual_reference_extra_pct"] = np.nan

MATCHER_ORDER = get_matcher_order(df)
MATCHER_COLOR_MAP = make_matcher_color_map(MATCHER_ORDER)

print("Consistent matcher colours:")
for matcher in MATCHER_ORDER:
    print(f"  {matcher}: {MATCHER_COLOR_MAP[matcher]}")

# ---------------------------------------------------------------------
# 3. Summary statistics
# ---------------------------------------------------------------------

def wilson_interval(k, n, z=1.96):
    """95% Wilson confidence interval for a binomial proportion."""
    if n <= 0:
        return np.nan, np.nan

    p_hat = k / n
    denom = 1 + z**2 / n
    centre = (p_hat + z**2 / (2 * n)) / denom
    half_width = z * np.sqrt(
        p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)
    ) / denom

    return max(0.0, centre - half_width), min(1.0, centre + half_width)


def make_segmentation_probability_summary(df):
    summary = (
        df
        .groupby(["error_mode", "error_pct", "perturbation_target", "matcher"], dropna=False)
        .agg(
            n_trials=("ssim_success", "size"),
            n_success=("ssim_success", "sum"),
            success_probability=("ssim_success", "mean"),
            median_ssim=("ssim", "median"),
            transform_rate=("has_transform", "mean"),
            median_actual_query_extra_pct=("actual_query_extra_pct", "median"),
            median_actual_reference_extra_pct=("actual_reference_extra_pct", "median"),
        )
        .reset_index()
    )

    ci = summary.apply(
        lambda r: wilson_interval(r["n_success"], r["n_trials"]),
        axis=1,
        result_type="expand",
    )

    summary["success_ci_low"] = ci[0]
    summary["success_ci_high"] = ci[1]

    return summary


df_prob = make_segmentation_probability_summary(df)

# ---------------------------------------------------------------------
# 4. Biology-friendly helper functions
# ---------------------------------------------------------------------

def pretty_error_mode(error_mode):
    if error_mode == "missed_nuclei":
        return "Missed nuclei"
    if error_mode == "extra_detections":
        return "Extra detections"
    return str(error_mode).replace("_", " ").title()


def x_label(error_mode):
    if error_mode == "missed_nuclei":
        return "True nuclei missed during segmentation (%)"
    if error_mode == "extra_detections":
        return "Extra/spurious detections added (%)"
    return "Segmentation error level (%)"


def short_bio_description(error_mode):
    if error_mode == "missed_nuclei":
        return (
            "False negatives: real nuclei were removed from the centroid set. "
            "This mimics weak nuclear labels, failed detections or overly conservative segmentation."
        )
    if error_mode == "extra_detections":
        return (
            "False positives: extra centroids were added. "
            "This mimics debris, background detections or over-calling of nuclear objects."
        )
    return ""


def observed_error_tolerance(summary, target_prob=0.90):
    """
    Estimate the largest tested error percentage at which the observed success
    probability remains above target_prob.

    This avoids over-interpreting fitted curves.
    """
    rows = []

    for (error_mode, perturbation_target, matcher), g in summary.groupby(
        ["error_mode", "perturbation_target", "matcher"],
        dropna=False,
    ):
        g = g.sort_values("error_pct")
        above = g[g["success_probability"] >= target_prob]

        if len(above) > 0:
            max_error = float(above["error_pct"].max())
            status = "observed"
        else:
            max_error = np.nan
            status = "below target at all tested levels"

        rows.append({
            "error_mode": error_mode,
            "perturbation_target": perturbation_target,
            "matcher": matcher,
            f"max_tested_error_with_at_least_{int(target_prob * 100)}pct_success": max_error,
            "status": status,
        })

    return pd.DataFrame(rows)


df_tolerance = observed_error_tolerance(df_prob, target_prob=TARGET_PROB)

# ---------------------------------------------------------------------
# 5. Axis helpers
# ---------------------------------------------------------------------

def get_categorical_x(summary, error_mode):
    """
    For extra detections, use equal spacing between the tested error levels.
    This avoids the awkward behaviour of symlog near zero and makes the plot
    easier to read. Tick labels still show the real percentages.
    """
    error_levels = sorted(
        summary.loc[summary["error_mode"] == error_mode, "error_pct"]
        .dropna()
        .unique()
    )

    level_to_pos = {level: i for i, level in enumerate(error_levels)}
    return error_levels, level_to_pos


def apply_x_axis(ax, error_mode, summary):
    if error_mode == "extra_detections":
        error_levels, _ = get_categorical_x(summary, error_mode)
        ax.set_xticks(range(len(error_levels)))
        ax.set_xticklabels([f"{int(v)}" for v in error_levels])
        ax.set_xlim(-0.3, len(error_levels) - 0.7)

        ax.text(
            0.5,
            -0.28,
            "Extra-detection levels are shown as tested categories.",
            transform=ax.transAxes,
            ha="center",
            va="top",
            fontsize=8,
            color="0.35",
        )
    else:
        xmax = np.nanmax(
            summary.loc[summary["error_mode"] == error_mode, "error_pct"].to_numpy(dtype=float)
        )
        ax.set_xlim(0, xmax * 1.05)


# ---------------------------------------------------------------------
# 6. Plot functions
# ---------------------------------------------------------------------

def plot_success_probability_panel(
    ax,
    summary,
    *,
    error_mode,
    perturbation_target,
    show_title=True,
    show_legend=True,
):
    d = summary[
        (summary["error_mode"] == error_mode) &
        (summary["perturbation_target"] == perturbation_target)
    ].copy()

    if d.empty:
        ax.text(
            0.5,
            0.5,
            f"No data for\n{error_mode}, {perturbation_target}",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_axis_off()
        return

    if error_mode == "extra_detections":
        error_levels, level_to_pos = get_categorical_x(summary, error_mode)

    for matcher in MATCHER_ORDER:
        if matcher not in set(d["matcher"].dropna().unique()):
            continue

        g = d[d["matcher"] == matcher].copy().sort_values("error_pct")
        color = MATCHER_COLOR_MAP[matcher]

        x_raw = g["error_pct"].to_numpy(dtype=float)
        if error_mode == "extra_detections":
            x = np.array([level_to_pos[v] for v in x_raw], dtype=float)
        else:
            x = x_raw

        y = g["success_probability"].to_numpy(dtype=float)
        lo = g["success_ci_low"].to_numpy(dtype=float)
        hi = g["success_ci_high"].to_numpy(dtype=float)

        ax.plot(
            x,
            y,
            marker="o",
            color=color,
            linewidth=2.0,
            markersize=5,
            label=matcher,
        )

        ax.fill_between(
            x,
            lo,
            hi,
            alpha=0.18,
            color=color,
            linewidth=0,
        )

    ax.axhline(
        TARGET_PROB,
        linestyle="--",
        linewidth=1.0,
        color="0.45",
        alpha=0.9,
    )

    ax.text(
        0.02,
        TARGET_PROB + 0.035,
        f"{int(TARGET_PROB * 100)}% success",
        transform=ax.get_yaxis_transform(),
        fontsize=9,
        color="0.35",
    )

    ax.set_ylim(-0.04, 1.04)
    ax.set_xlabel(x_label(error_mode))
    ax.set_ylabel("Probability of correct localisation\n(SSIM-based success)")

    apply_x_axis(ax, error_mode, summary)

    if show_title:
        if perturbation_target == "query":
            title_suffix = "query segmentation corrupted"
        elif perturbation_target == "both":
            title_suffix = "query and reference corrupted"
        else:
            title_suffix = f"{perturbation_target} corrupted"

        ax.set_title(f"{pretty_error_mode(error_mode)}\n{title_suffix}")

    ax.grid(True, alpha=0.25)

    if show_legend:
        ax.legend(frameon=False, title="Matcher", bbox_to_anchor=(1.02, 1), loc="upper left")


def plot_main_segmentation_error_figure(summary, perturbation_target="query"):
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), dpi=150)

    plot_success_probability_panel(
        axes[0],
        summary,
        error_mode="missed_nuclei",
        perturbation_target=perturbation_target,
        show_title=True,
        show_legend=False,
    )

    plot_success_probability_panel(
        axes[1],
        summary,
        error_mode="extra_detections",
        perturbation_target=perturbation_target,
        show_title=True,
        show_legend=True,
    )

    fig.suptitle(
        "Robustness of NucleiSky localisation to segmentation errors",
        y=1.05,
        fontsize=13,
    )

    fig.tight_layout()

    out_path = plot_output_dir / f"segmentation_error_success_probability_{perturbation_target}.pdf"
    fig.savefig(out_path, bbox_inches="tight", transparent=True)
    print(f"Saved: {out_path}")

    plt.show()


def plot_median_ssim_figure(summary, perturbation_target="query"):
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), dpi=150)

    for ax, mode in zip(axes, ["missed_nuclei", "extra_detections"]):
        d = summary[
            (summary["error_mode"] == mode) &
            (summary["perturbation_target"] == perturbation_target)
        ].copy()

        if d.empty:
            ax.set_axis_off()
            continue

        if mode == "extra_detections":
            error_levels, level_to_pos = get_categorical_x(summary, mode)

        for matcher in MATCHER_ORDER:
            if matcher not in set(d["matcher"].dropna().unique()):
                continue

            g = d[d["matcher"] == matcher].copy().sort_values("error_pct")
            color = MATCHER_COLOR_MAP[matcher]

            x_raw = g["error_pct"].to_numpy(dtype=float)
            if mode == "extra_detections":
                x = np.array([level_to_pos[v] for v in x_raw], dtype=float)
            else:
                x = x_raw

            ax.plot(
                x,
                g["median_ssim"],
                marker="o",
                color=color,
                linewidth=2.0,
                markersize=5,
                label=matcher,
            )

        ax.axhline(
            SSIM_SUCCESS_THRESH,
            linestyle="--",
            linewidth=1.0,
            color="0.45",
            alpha=0.9,
        )

        ax.set_ylim(-0.04, 1.04)
        ax.set_xlabel(x_label(mode))
        ax.set_ylabel("Median SSIM")
        ax.set_title(pretty_error_mode(mode))
        ax.grid(True, alpha=0.25)

        apply_x_axis(ax, mode, summary)

    axes[1].legend(frameon=False, title="Matcher", bbox_to_anchor=(1.02, 1), loc="upper left")

    fig.suptitle(
        "Image-level validation of predicted localisation",
        y=1.05,
        fontsize=13,
    )

    fig.tight_layout()

    out_path = plot_output_dir / f"segmentation_error_median_ssim_{perturbation_target}.pdf"
    fig.savefig(out_path, bbox_inches="tight", transparent=True)
    print(f"Saved: {out_path}")

    plt.show()


# ---------------------------------------------------------------------
# 7. Generate plots
# ---------------------------------------------------------------------

display(Markdown(
    "## Segmentation-error robustness\n"
    "This benchmark asks how much segmentation error NucleiSky can tolerate before it fails to recover the correct field of view. "
    "Success is defined only by image agreement between the original query crop and the predicted reference crop "
    f"(**SSIM ≥ {SSIM_SUCCESS_THRESH:.2f}**). Matcher-reported success and inlier fraction are diagnostics only."
))

display(Markdown(
    "### Error modes\n"
    f"- **Missed nuclei:** {short_bio_description('missed_nuclei')}\n"
    f"- **Extra detections:** {short_bio_description('extra_detections')}\n\n"
    "The **query-only** condition is the cleanest interpretation because only the query segmentation is degraded. "
    "The **query + reference** condition is a harsher stress test. "
    "Matcher colours are fixed across all plots."
))

# Main query-only plots
plot_main_segmentation_error_figure(df_prob, perturbation_target=MAIN_TARGET)
plot_median_ssim_figure(df_prob, perturbation_target=MAIN_TARGET)

# Optional harsher stress test
if PLOT_BOTH_TARGET and "both" in set(df_prob["perturbation_target"]):
    plot_main_segmentation_error_figure(df_prob, perturbation_target="both")
    plot_median_ssim_figure(df_prob, perturbation_target="both")

# ---------------------------------------------------------------------
# 8. Display biology-friendly summary tables
# ---------------------------------------------------------------------

summary_display = df_prob.copy()

summary_display["error_type"] = summary_display["error_mode"].map(pretty_error_mode)
summary_display["condition"] = summary_display["perturbation_target"].map({
    "query": "Query segmentation corrupted",
    "both": "Query and reference corrupted",
    "reference": "Reference segmentation corrupted",
}).fillna(summary_display["perturbation_target"])

summary_display = summary_display[
    [
        "error_type",
        "condition",
        "matcher",
        "error_pct",
        "n_trials",
        "n_success",
        "success_probability",
        "success_ci_low",
        "success_ci_high",
        "median_ssim",
        "transform_rate",
    ]
].sort_values(["condition", "error_type", "matcher", "error_pct"])

display(Markdown("### Per-condition success summary"))
display(
    summary_display.style
    .format({
        "error_pct": "{:.0f}",
        "success_probability": "{:.2f}",
        "success_ci_low": "{:.2f}",
        "success_ci_high": "{:.2f}",
        "median_ssim": "{:.2f}",
        "transform_rate": "{:.2f}",
    })
    .background_gradient(
        subset=["success_probability"],
        cmap="Greens",
        vmin=0,
        vmax=1,
    )
)

tolerance_display = df_tolerance.copy()
tolerance_display["error_type"] = tolerance_display["error_mode"].map(pretty_error_mode)
tolerance_display["condition"] = tolerance_display["perturbation_target"].map({
    "query": "Query segmentation corrupted",
    "both": "Query and reference corrupted",
    "reference": "Reference segmentation corrupted",
}).fillna(tolerance_display["perturbation_target"])

tolerance_col = f"max_tested_error_with_at_least_{int(TARGET_PROB * 100)}pct_success"

tolerance_display = tolerance_display[
    [
        "error_type",
        "condition",
        "matcher",
        tolerance_col,
        "status",
    ]
].sort_values(["condition", "error_type", "matcher"])

display(Markdown(f"### Largest tested segmentation-error level with ≥{int(TARGET_PROB * 100)}% success"))
display(
    tolerance_display.style
    .format({
        tolerance_col: "{:.0f}",
    })
    .background_gradient(
        subset=[tolerance_col],
        cmap="Blues",
    )
)

# Save tables
summary_table_path = plot_output_dir / "segmentation_error_success_summary_biofriendly.csv"
tolerance_table_path = plot_output_dir / "segmentation_error_tolerance_summary_biofriendly.csv"

summary_display.to_csv(summary_table_path, index=False)
tolerance_display.to_csv(tolerance_table_path, index=False)

print(f"Saved summary table: {summary_table_path}")
print(f"Saved tolerance table: {tolerance_table_path}")

# ---------------------------------------------------------------------
# 9. Automatic plain-language interpretation
# ---------------------------------------------------------------------

def best_matcher_sentence(tolerance_display, condition="Query segmentation corrupted"):
    rows = []

    for error_type in ["Missed nuclei", "Extra detections"]:
        sub = tolerance_display[
            (tolerance_display["condition"] == condition) &
            (tolerance_display["error_type"] == error_type)
        ].copy()

        if sub.empty:
            continue

        valid = sub[np.isfinite(sub[tolerance_col])]
        if valid.empty:
            rows.append(
                f"- For **{error_type.lower()}**, no matcher maintained ≥{int(TARGET_PROB * 100)}% success across the tested error levels."
            )
            continue

        best = valid.sort_values(tolerance_col, ascending=False).iloc[0]
        rows.append(
            f"- For **{error_type.lower()}**, the most tolerant matcher was **{best['matcher']}**, "
            f"remaining above ≥{int(TARGET_PROB * 100)}% success up to the largest tested level of "
            f"**{best[tolerance_col]:.0f}%**."
        )

    return "\n".join(rows)

display(Markdown(
    "### Plain-language interpretation\n"
    "Use the query-only plots for the main biological interpretation. They answer how robust localisation is when the segmentation of the field of view is imperfect, while the reference remains unchanged.\n\n"
    + best_matcher_sentence(tolerance_display, condition="Query segmentation corrupted")
    + "\n\n"
    "The query + reference condition is more conservative because errors are introduced on both sides of the match. "
    "This is useful as a stress test, but it is less specific about whether failure is driven by the query segmentation or by corruption of the reference constellation."
))